# DR-VERGE — Final Research Notebook

**A rigorous investigation of complementarity-aware knowledge transfer and INT8 deployment for
lightweight two-field diabetic retinopathy grading.**

Implements `revision/dr-verge-rev.md` on top of the validated rev3 core. Supersedes
`full_pipeline_notebook_rev3.ipynb`.

---

## Research questions (locked before running)

**RQ1 — Knowledge transfer.** *To what extent can Complementarity-Shift Distillation transfer the
dual-view decision benefit of a two-field teacher to a lightweight student, compared with no
distillation, standard logit distillation, and feature distillation?*

Judged on **two independent axes**, so the finding is informative regardless of which way it lands:
- *Predictive*: QWK (primary), Accuracy, Macro-F1, MAE, Severe-Error Rate
- *Mechanistic*: ShiftMAE, Cosine agreement, Benefit correlation, internal/external dual-view gain

If QWK(CSD) ≈ QWK(KD) but ShiftFidelity(CSD) > ShiftFidelity(KD), that is still a scientific
finding. If CSD fails on both, that is a valid answer too.

**RQ2 — Quantization / deployment.** *To what extent can INT8 post-training quantization and
quantization-aware training reduce model size and CPU latency while preserving categorical and
ordinal grading performance of the best lightweight dual-view model?*

Compares `M*_FP32` vs `M*_PTQ-INT8` vs `M*_QAT-INT8`, where `M*` is selected **on validation only**.

---

## Locked protocol (do not change after the first full run)

| Item | Value |
|---|---|
| Primary metric | **QWK** (ordinal; DR grades are 0<1<2<3<4) |
| Core seeds | 42, 123, 2026, 3407, 8888 (**5**) |
| Baseline seeds | 42, 123, 2026 (**3**) |
| Model selection | `argmax QWK_val`; ties (<0.005) → Macro-F1 → lower SER → lower MAE → simpler method |
| Test set | DRTiD official test — touched **once**, after selection |
| External validation | DeepDRiD — frozen, no tuning, evaluated last |
| Statistics | Paired patient-clustered bootstrap, B=10,000, Holm-corrected |
| Pre-registered comparisons | RQ1: CSD vs {NoDistill, LogitKD, FeatureKD}. RQ2: {PTQ, QAT} vs FP32, QAT vs PTQ |

**Everything is saved.** Every figure ships PNG+PDF+SVG **and** a companion CSV — no number lives
only inside an image. Per-sample predictions, per-epoch gradient contributions, configs, metadata,
and a model registry are all written to disk.

---

## What this adds over rev3

rev3 fixed the three defects that made rev2's RQ1 test uninformative (collapsed CORAL thresholds,
40×-undersized student, CSD with no gradient). That core is **kept unchanged**. This notebook adds:

1. 5 seeds on core conditions (was 3)
2. Complete categorical metrics: Accuracy, Balanced Accuracy, macro/weighted P/R/F1, per-grade
   P/R/F1/specificity/support
3. Confusion matrices (raw + normalized) with **automatic prediction-collapse warnings**
4. **QAT** alongside PTQ — RQ2 becomes a three-way FP32/PTQ/QAT comparison
5. Modern quantization + export: PT2E/torchao primary, eager fallback, `torch.export` + ONNX
   (TorchScript is deprecated and is no longer the deployment path)
6. Full efficiency suite: params, serialized size, compression ratio, mean/median/p95/p99 latency,
   throughput, speedup, memory — under a standardized benchmark protocol
7. Performance-retention metrics (INT8 vs FP32)
8. Statistics: paired patient-clustered bootstrap B=10,000, Holm correction, effect sizes with CIs
9. DeepDRiD **external confirmatory validation**, frozen
10. Deployment artifacts + `predict_dr()` inference wrapper + parity checks + model registry
11. Ten publication-grade figures, each with a companion data CSV
12. Gates 0–9 with a final consolidated gate report

## 01 — Environment & Reproducibility (Gate 0)

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab ships a CUDA-enabled torch -- we deliberately do NOT reinstall it. torchao is needed for
# the modern PT2E quantization path (PTQ + QAT); if it is unavailable the notebook falls back to
# eager-mode quantization and SAYS SO explicitly rather than pretending the modern path ran.
!pip install -q albumentations scikit-learn pandas tqdm pyyaml psutil onnx onnxruntime
!pip install -q torchao || echo "torchao unavailable -- will use eager-mode quantization fallback"

import torch, torchvision, numpy, sklearn, platform, subprocess, json, os
print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("numpy       :", numpy.__version__)
print("sklearn     :", sklearn.__version__)
try:
    import torchao; print("torchao     :", torchao.__version__)
except Exception as e:
    print("torchao     : NOT AVAILABLE ->", e)
print("python      :", platform.python_version())
print("CUDA avail  :", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("quant engines:", torch.backends.quantized.supported_engines)
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > GPU."

## 02–03 — Locked configuration & paths

In [ ]:
import os, json, hashlib, platform, subprocess, math, random, time, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

# ---------------- EDIT THIS ONE LINE ----------------
DRIVE_BASE = "/content/drive/MyDrive/DR-VERGE"
# ----------------------------------------------------

DATASET_ROOT = f"{DRIVE_BASE}/dataset"

def _resolve_drtid_root(root):
    for cand in (f"{root}/DRTiD/DRTiD", f"{root}/DRTiD"):
        if os.path.exists(f"{cand}/Ground Truths/DR_grade/a. DR_grade_Training.csv"):
            return cand
    return f"{root}/DRTiD/DRTiD"

def _resolve_deepdrid_root(root):
    for cand in (f"{root}/DeepDRiD-master/regular_fundus_images",
                 f"{root}/DeepDRiD/regular_fundus_images",
                 f"{root}/DeepDRiD-master", f"{root}/DeepDRiD"):
        if os.path.exists(f"{cand}/regular-fundus-validation/regular-fundus-validation.csv"):
            return cand
    return None

DRTID_ROOT    = _resolve_drtid_root(DATASET_ROOT)
APTOS_ROOT    = f"{DATASET_ROOT}/APTOS"
DEEPDRID_ROOT = _resolve_deepdrid_root(DATASET_ROOT)

ART          = f"{DRIVE_BASE}/artifacts"
SPLITS_DIR   = f"{ART}/splits"
CKPT_DIR     = f"{ART}/checkpoints"
MODELS_DIR   = f"{ART}/models"
RESULTS_DIR  = f"{ART}/results"
FIGURES_DIR  = f"{RESULTS_DIR}/figures"
TABLES_DIR   = f"{RESULTS_DIR}/tables"
METRICS_DIR  = f"{RESULTS_DIR}/metrics"
PREDS_DIR    = f"{RESULTS_DIR}/predictions"
LOGS_DIR     = f"{RESULTS_DIR}/logs"
CONFIG_DIR   = f"{ART}/configs"
for d in [ART, SPLITS_DIR, CKPT_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
          METRICS_DIR, PREDS_DIR, LOGS_DIR, CONFIG_DIR,
          f"{CKPT_DIR}/pretrained_backbones", f"{CKPT_DIR}/teacher", f"{CKPT_DIR}/student"]:
    os.makedirs(d, exist_ok=True)

# ================= LOCKED EXPERIMENT PROTOCOL =================
SEEDS_CORE     = [42, 123, 2026, 3407, 8888]   # no-distill / logit-KD / feature-KD / CSD
SEEDS_BASELINE = [42, 123, 2026]               # single-view baselines, ablations, QAT
PRIMARY_SEED   = 42

IMG_SIZE       = 224
NUM_CLASSES    = 5
NUM_THRESHOLDS = NUM_CLASSES - 1
POS_WEIGHT_MODE   = "sqrt"                                  # none | sqrt | full
STUDENT_CHANNELS  = (32, 64, 96, 128, 160, 192, 224)        # ~330K-param student
FUSION_TYPE       = "interaction_mlp"

# Model selection (validation only) -- tie-break chain fixed in advance
SELECTION_METRIC   = "QWK"
SELECTION_TIE_EPS  = 0.005
SELECTION_TIEBREAK = ["MacroF1", "-SevereErrorRate", "-MAE"]

# Statistics
BOOTSTRAP_B      = 10000
BOOTSTRAP_ALPHA  = 0.05
PREREGISTERED_COMPARISONS = {
    "RQ1": [("dual_csd", "dual_no_distill"), ("dual_csd", "dual_logitkd"), ("dual_csd", "dual_featkd")],
    "RQ2": [("ptq_int8", "best_fp32"), ("qat_int8", "best_fp32"), ("qat_int8", "ptq_int8")],
}

# Standardized CPU benchmark protocol
BENCH = {"batch_size": 1, "warmup": 50, "runs": 500, "threads": 1}

# DeepDRiD field-order is NOT documented in its public CSVs (no column says which of _1/_2 is
# macula- vs disc-centred). Rather than hide that behind an assumption, external validation is
# evaluated under BOTH orderings and both are reported -- turning the unknown into a robustness check.
DEEPDRID_FIELD_ORDERS = ["_1=macula", "_1=disc"]

CONFIG_SNAPSHOT = dict(
    seeds_core=SEEDS_CORE, seeds_baseline=SEEDS_BASELINE, img_size=IMG_SIZE,
    num_classes=NUM_CLASSES, pos_weight_mode=POS_WEIGHT_MODE,
    student_channels=list(STUDENT_CHANNELS), fusion_type=FUSION_TYPE,
    selection_metric=SELECTION_METRIC, selection_tie_eps=SELECTION_TIE_EPS,
    selection_tiebreak=SELECTION_TIEBREAK, bootstrap_B=BOOTSTRAP_B, bench=BENCH,
)

_expected = [f"{DRTID_ROOT}/Ground Truths/DR_grade/a. DR_grade_Training.csv",
             f"{DRTID_ROOT}/Ground Truths/DR_grade/b. DR_grade_Testing.csv",
             f"{DRTID_ROOT}/Original Images",
             f"{APTOS_ROOT}/train_1.csv", f"{APTOS_ROOT}/valid.csv",
             f"{APTOS_ROOT}/train_images/train_images", f"{APTOS_ROOT}/val_images/val_images"]
_missing = [p for p in _expected if not os.path.exists(p)]
if _missing:
    raise FileNotFoundError("Dataset missing:\n" + "\n".join(_missing))

print("DRTID_ROOT    :", DRTID_ROOT)
print("APTOS_ROOT    :", APTOS_ROOT)
print("DEEPDRID_ROOT :", DEEPDRID_ROOT or "NOT FOUND -- external validation will be SKIPPED (reported, not hidden)")
print("artifacts     :", ART)

In [ ]:
# ---- Gate 0: environment lock + provenance ----
def _pip_freeze():
    try:
        return subprocess.check_output(["pip", "freeze"], text=True)
    except Exception as e:
        return f"(pip freeze failed: {e})"

def _git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True,
                                        stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unavailable (not a git checkout in this runtime)"

import torchvision, sklearn
try:
    import torchao; _torchao_v = torchao.__version__
except Exception:
    _torchao_v = None

ENVIRONMENT = {
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "torchao": _torchao_v, "numpy": np.__version__, "sklearn": sklearn.__version__,
    "python": platform.python_version(), "platform": platform.platform(),
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "quantized_engines": list(torch.backends.quantized.supported_engines),
    "git_commit": _git_commit(), "timestamp": pd.Timestamp.now().isoformat(),
}
with open(f"{CONFIG_DIR}/environment.json", "w") as f:
    json.dump(ENVIRONMENT, f, indent=2)
with open(f"{CONFIG_DIR}/pip_freeze.txt", "w") as f:
    f.write(_pip_freeze())
with open(f"{CONFIG_DIR}/config_locked.json", "w") as f:
    json.dump(CONFIG_SNAPSHOT, f, indent=2)

GATES = {}
def record_gate(name, passed, detail=""):
    GATES[name] = {"passed": bool(passed), "detail": detail}
    print(f"{'PASS' if passed else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    return passed

record_gate("Gate0_Environment", True,
            f"torch={torch.__version__} torchao={_torchao_v} engines={ENVIRONMENT['quantized_engines']}")
print(json.dumps(ENVIRONMENT, indent=2))

## 04 — Reproducibility utilities

In [ ]:
def set_seed(seed: int, deterministic: bool = False):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

def make_generator(seed):
    g = torch.Generator(); g.manual_seed(seed); return g

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def robust_torch_load(path, map_location=None, retries=6, delay=1.0):
    """Drive's FUSE mount can lag behind its own writes -- retry rather than crash a long run."""
    last = None
    for i in range(retries):
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except (FileNotFoundError, OSError) as e:
            last = e
            if i < retries - 1:
                print(f"  robust_torch_load retry {i+1}/{retries} for {path}")
                time.sleep(delay); delay *= 1.5
    raise last

def robust_torch_save(obj, path, retries=6, delay=1.0):
    last = None
    parent = os.path.dirname(path)
    for i in range(retries):
        try:
            if parent: os.makedirs(parent, exist_ok=True)
            torch.save(obj, path); return
        except (RuntimeError, OSError) as e:
            last = e
            if i < retries - 1:
                print(f"  robust_torch_save retry {i+1}/{retries} for {path}: {e}")
                time.sleep(delay); delay *= 1.5
    raise last

def checkpoint_is_compatible(ckpt_path, model, unwrap_key="model_state"):
    """Side-effect-free key/shape check -- never partially mutates `model`."""
    if not os.path.exists(ckpt_path): return False
    try:
        raw = robust_torch_load(ckpt_path, map_location="cpu")
        state = raw[unwrap_key] if (unwrap_key and isinstance(raw, dict) and unwrap_key in raw) else raw
        cur = model.state_dict()
        if set(state.keys()) != set(cur.keys()):
            miss = list(set(cur) - set(state))[:4]; unexp = list(set(state) - set(cur))[:4]
            raise RuntimeError(f"key mismatch missing={miss} unexpected={unexp}")
        for k in state:
            if state[k].shape != cur[k].shape:
                raise RuntimeError(f"shape mismatch '{k}': {tuple(state[k].shape)} vs {tuple(cur[k].shape)}")
        return True
    except Exception as e:
        print(f"  {os.path.basename(ckpt_path)} incompatible with current architecture ({e}) -- retraining.")
        return False

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f: json.dump(obj, f, indent=2, default=str)

print("Reproducibility utilities defined.")

## 05 — DRTiD integrity, splits & exploratory statistics (Gate 1)

DRTiD ships an **official** train/test split, used as-is. We only carve train/val out of the
official training rows. `_1` = Macula, `_2` = Optic disc — confirmed against the CrossFiT
reference loader (DRTiD's own benchmark authors' code).

**Scope note (verified, not assumed):** every `ID` in DRTiD's ground truth appears exactly once and
none carries both an `L` and `R` row, so `ID` is a per-**eye** identifier with no patient linkage
exposed. Splits and bootstrap clustering group by `ID` because it is the finest key the data
provides — that is eye-wise, *not* verified patient-wise. Reported as a limitation, not papered over.

In [ ]:
from sklearn.model_selection import train_test_split

def make_drtid_splits(seed=42, val_fraction=0.2, force=False):
    out = {k: f"{SPLITS_DIR}/drtid_{k}.csv" for k in ("train", "val", "test")}
    images_dir = f"{DRTID_ROOT}/Original Images"
    off_train = pd.read_csv(f"{DRTID_ROOT}/Ground Truths/DR_grade/a. DR_grade_Training.csv")
    off_test  = pd.read_csv(f"{DRTID_ROOT}/Ground Truths/DR_grade/b. DR_grade_Testing.csv")

    overlap = set(off_train["ID"]) & set(off_test["ID"])
    assert not overlap, f"Gate 1 FAILED: official train/test share IDs: {sorted(overlap)[:10]}"

    def std(df):
        return pd.DataFrame({
            "patient_id": df["ID"],
            "macula_path": df["Macula"].apply(lambda s: f"{images_dir}/{s}.jpg"),
            "disc_path":   df["Optic disc"].apply(lambda s: f"{images_dir}/{s}.jpg"),
            "grade":       df["Grade"],
            "laterality":  df["LR"],
        })

    if not force and all(os.path.exists(p) for p in out.values()):
        print("Splits already exist on Drive -- reusing (guarantees identical splits across sessions).")
    else:
        tr_ids, va_ids = train_test_split(off_train["ID"].values, test_size=val_fraction, random_state=seed)
        std(off_train[off_train["ID"].isin(tr_ids)]).to_csv(out["train"], index=False)
        std(off_train[off_train["ID"].isin(va_ids)]).to_csv(out["val"], index=False)
        std(off_test).to_csv(out["test"], index=False)

    dfs = {k: pd.read_csv(v) for k, v in out.items()}
    assert not (set(dfs["train"].patient_id) & set(dfs["val"].patient_id)), "Gate 1 FAILED: train/val overlap"
    assert not (set(dfs["val"].patient_id) & set(dfs["test"].patient_id)),  "Gate 1 FAILED: val/test overlap"
    assert not (set(dfs["train"].patient_id) & set(dfs["test"].patient_id)), "Gate 1 FAILED: train/test overlap"

    rows, ok = [], True
    for name, df in dfs.items():
        missing = [p for c in ("macula_path", "disc_path") for p in df[c] if not os.path.exists(p)]
        if missing:
            ok = False; print(f"  MISSING {len(missing)} images in {name}, e.g. {missing[:3]}")
        dist = df["grade"].value_counts().sort_index()
        absent = sorted(set(range(NUM_CLASSES)) - set(dist.index))
        if absent:
            ok = False; print(f"  {name}: grades {absent} ABSENT")
        rows.append({"split": name, "n_eyes": len(df), "n_images": 2 * len(df),
                     **{f"grade_{g}": int(dist.get(g, 0)) for g in range(NUM_CLASSES)}})
    stats = pd.DataFrame(rows)
    stats.to_csv(f"{TABLES_DIR}/table_00_dataset_statistics.csv", index=False)
    print(stats.to_string(index=False))

    manifest = {k: {"path": v, "sha256": sha256_file(v), "rows": len(dfs[k])} for k, v in out.items()}
    save_json(manifest, f"{CONFIG_DIR}/split_manifest.json")
    record_gate("Gate1_Dataset", ok, f"train/val/test = {len(dfs['train'])}/{len(dfs['val'])}/{len(dfs['test'])} eyes; "
                                     f"no ID overlap; all grades present; all images resolve")
    return out["train"], out["val"], out["test"]

DRTID_TRAIN_CSV, DRTID_VAL_CSV, DRTID_TEST_CSV = make_drtid_splits()

## 06 — Preprocessing & augmentation

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# DRTiD channel stats from the CrossFiT authors' own loader -- keeps preprocessing aligned with
# the benchmark this work is positioned against.
DRTID_MEAN, DRTID_STD = [0.372487, 0.217266, 0.119367], [0.281526, 0.179457, 0.109162]
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def build_transforms(train, mean, std):
    # Horizontal flip deliberately OMITTED: it risks changing macula/disc laterality semantics, and
    # the CrossFiT reference implementation has its flip code commented out for the same reason.
    ops = [A.Resize(IMG_SIZE, IMG_SIZE)]
    if train:
        ops += [A.Rotate(limit=15, p=0.7),
                A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5)]
    ops += [A.Normalize(mean=mean, std=std), ToTensorV2()]
    return A.Compose(ops)

train_transform = build_transforms(True,  DRTID_MEAN, DRTID_STD)
eval_transform  = build_transforms(False, DRTID_MEAN, DRTID_STD)
aptos_train_transform = build_transforms(True,  IMAGENET_MEAN, IMAGENET_STD)
aptos_eval_transform  = build_transforms(False, IMAGENET_MEAN, IMAGENET_STD)

PREPROCESSING_META = {"input_size": [IMG_SIZE, IMG_SIZE], "normalization_mean": DRTID_MEAN,
                      "normalization_std": DRTID_STD, "horizontal_flip": False,
                      "views": ["macula", "optic_disc"], "ordinal_threshold": 0.5}

def _rgb(path): return np.array(Image.open(path).convert("RGB"))

class DRTiDDualViewDataset(Dataset):
    def __init__(self, split_csv, transform=None):
        self.df = pd.read_csv(split_csv)
        self.transform = transform if transform is not None else eval_transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {"macula": self.transform(image=_rgb(r["macula_path"]))["image"],
                "disc":   self.transform(image=_rgb(r["disc_path"]))["image"],
                "label":  torch.tensor(int(r["grade"]), dtype=torch.long),
                "patient_id": int(r["patient_id"])}

class APTOSSingleViewDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform=None):
        self.df = pd.read_csv(csv_path); self.root_dir = root_dir
        self.transform = transform if transform is not None else aptos_eval_transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = _rgb("{}/{}.png".format(self.root_dir, r["id_code"]))
        return {"image": self.transform(image=img)["image"],
                "label": torch.tensor(int(r["diagnosis"]), dtype=torch.long)}

def make_loader(ds, batch_size, shuffle, seed=None, workers=2):
    kw = {}
    if seed is not None:
        kw = {"worker_init_fn": seed_worker, "generator": make_generator(seed)}
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=workers, **kw)

print("Preprocessing defined.")

## 07–09 — Model architecture, CORAL initialization & unit tests

**CORAL thresholds are initialized from the empirical marginal** `b_k = logit(P(Y>k))`. rev2
initialized all four thresholds within 0.15 logits of each other while DRTiD needs a 3.24-logit
spread; because CORAL gives each sample one scalar score compared against all thresholds, collapsed
thresholds make intermediate grades unreachable — measured rev2 sensitivity for Grades 1–3 was
0.00–0.04 across every condition including the teacher. Kept from rev3, with assertions.

In [ ]:
import torchvision.models as tv

def compute_pos_weights(train_csv, num_thresholds=NUM_THRESHOLDS, grade_col="grade", mode=POS_WEIGHT_MODE):
    """pos_weight_k = N_neg/N_pos. Raw ratio is 24.8x at k=3 on DRTiD (31/800 eyes are Grade 4),
    which drove rev2's collapse onto the extreme grades. 'sqrt' keeps the correction's direction
    without its degeneracy."""
    g = pd.read_csv(train_csv)[grade_col].values
    w = []
    for k in range(num_thresholds):
        pos, neg = int((g > k).sum()), int((g <= k).sum())
        if pos == 0 or neg == 0:
            raise ValueError(f"degenerate threshold k={k}: pos={pos} neg={neg}")
        r = neg / pos
        w.append({"full": r, "sqrt": math.sqrt(r), "none": 1.0}[mode])
    return torch.tensor(w, dtype=torch.float32)

def compute_init_thresholds(train_csv, num_thresholds=NUM_THRESHOLDS, grade_col="grade", eps=1e-3):
    g = pd.read_csv(train_csv)[grade_col].values
    return [math.log(min(max(float((g > k).mean()), eps), 1 - eps) /
                     (1 - min(max(float((g > k).mean()), eps), 1 - eps))) for k in range(num_thresholds)]


class CORALHead(nn.Module):
    """Monotone cumulative outputs P(y>k) by construction (ordered non-negative softplus steps)."""
    def __init__(self, in_dim, num_classes=NUM_CLASSES, init_thresholds=None):
        super().__init__()
        self.num_thresholds = num_classes - 1
        self.fc = nn.Linear(in_dim, 1, bias=False)
        if init_thresholds is None:
            init_thresholds = [-0.7 * i for i in range(self.num_thresholds)]
        t = torch.tensor(list(init_thresholds), dtype=torch.float32)
        if t.numel() != self.num_thresholds:
            raise ValueError(f"need {self.num_thresholds} thresholds, got {t.numel()}")
        gaps = (t[:-1] - t[1:]).clamp_min(1e-4)
        self.base_bias  = nn.Parameter(t[0].clone())
        self.bias_steps = nn.Parameter(torch.log(torch.expm1(gaps)).clone())

    def _ordered_biases(self):
        steps = F.softplus(self.bias_steps)
        cum = torch.cat([torch.zeros(1, device=steps.device), torch.cumsum(steps, dim=0)])
        return self.base_bias - cum

    def forward(self, z):
        logits = self.fc(z) + self._ordered_biases().unsqueeze(0)
        return logits, torch.sigmoid(logits)


class InteractionFusion(nn.Module):
    """Concat + |diff| + product through a small MLP (judge.md Flag 2: a bare linear fusion can
    only form a weighted sum and cannot represent cross-view interaction). LayerNorm rather than
    BatchNorm removes batch-size sensitivity at the small batch sizes used here.

    All submodules are defined unconditionally: TorchScript/export statically analyses every branch,
    and rev2's Gate 5 failed with "has no attribute 'norm'" because submodules were created only
    inside one branch of an if."""
    def __init__(self, feat_dim, fusion_type=FUSION_TYPE, hidden_dim=None):
        super().__init__()
        if fusion_type not in ("linear", "interaction_mlp"):
            raise ValueError(fusion_type)
        self.fusion_type = fusion_type
        hidden_dim = hidden_dim or feat_dim
        self.norm     = nn.LayerNorm(feat_dim * 2)
        self.norm_in  = nn.LayerNorm(feat_dim * 4)
        self.proj     = nn.Linear(feat_dim * 4, hidden_dim)
        self.act      = nn.ReLU(inplace=True)
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.out_dim  = feat_dim * 2 if fusion_type == "linear" else hidden_dim

    def forward(self, z_m, z_d):
        if self.fusion_type == "linear":
            return self.norm(torch.cat([z_m, z_d], dim=1))
        combined = self.norm_in(torch.cat([z_m, z_d, torch.abs(z_m - z_d), z_m * z_d], dim=1))
        return self.norm_out(self.act(self.proj(combined)))


class DepthwiseSeparableBlock(nn.Module):
    """ReLU (not ReLU6): eager-mode fuse_modules has no fuser for Conv-BN-ReLU6."""
    def __init__(self, i, o, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(i, i, 3, stride=stride, padding=1, groups=i, bias=False)
        self.bn1 = nn.BatchNorm2d(i); self.act1 = nn.ReLU(inplace=True)
        self.pw = nn.Conv2d(i, o, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(o); self.act2 = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act2(self.bn2(self.pw(self.act1(self.bn1(self.dw(x))))))
    def fuse(self, qat=False):
        fn = torch.ao.quantization.fuse_modules_qat if qat else torch.ao.quantization.fuse_modules
        fn(self, [["dw", "bn1", "act1"], ["pw", "bn2", "act2"]], inplace=True)


class LightweightBackbone(nn.Module):
    """~125K-param feature extractor. rev2's was 8,176 params (its fusion MLP was 75% of the whole
    34K model), ~40x below the technical doc's 0.3-0.4M target, which capacity-capped every
    dual-view condition at the same QWK and made RQ1 untestable."""
    def __init__(self, channels=None):
        super().__init__()
        ch = tuple(channels or STUDENT_CHANNELS)
        self.stem_conv = nn.Conv2d(3, ch[0], 3, stride=2, padding=1, bias=False)
        self.stem_bn = nn.BatchNorm2d(ch[0]); self.stem_act = nn.ReLU(inplace=True)
        strides = [2 if i % 2 == 0 else 1 for i in range(len(ch) - 1)]
        self.blocks = nn.ModuleList([DepthwiseSeparableBlock(ch[i], ch[i+1], strides[i])
                                     for i in range(len(ch) - 1)])
        self.gap = nn.AdaptiveAvgPool2d(1); self.out_dim = ch[-1]
    def forward(self, x):
        x = self.stem_act(self.stem_bn(self.stem_conv(x)))
        for b in self.blocks: x = b(x)
        return self.gap(x).flatten(1)
    def fuse_model(self, qat=False):
        fn = torch.ao.quantization.fuse_modules_qat if qat else torch.ao.quantization.fuse_modules
        fn(self, [["stem_conv", "stem_bn", "stem_act"]], inplace=True)
        for b in self.blocks: b.fuse(qat=qat)


class _DualViewBase(nn.Module):
    def forward(self, macula, disc):
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        z_f = self.fusion(z_m, z_d)
        ld, pd_ = self.main_head(z_f)
        lm, pm = self.macula_head(z_m)
        ldd, pdd = self.disc_head(z_d)
        return {"p_dual": pd_, "logit_dual": ld, "p_macula": pm, "logit_macula": lm,
                "p_disc": pdd, "logit_disc": ldd, "z_fused": z_f}
    def forward_single(self, x, which="macula"):
        z = self.backbone(x)
        logit, p = (self.macula_head if which == "macula" else self.disc_head)(z)
        return {"logit": logit, "p": p}
    def counterfactual_forward(self, macula, disc):
        """Same-head counterfactual (judge.md Flag 1/3): dual / macula-only / disc-only all go
        through the SAME main_head, so their difference cannot be head discrepancy."""
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        zero = torch.zeros_like(z_m)
        _, p_dual = self.main_head(self.fusion(z_m, z_d))
        _, p_m    = self.main_head(self.fusion(z_m, zero))
        _, p_d    = self.main_head(self.fusion(zero, z_d))
        return {"p_dual": p_dual, "p_macula_cf": p_m, "p_disc_cf": p_d}


class DualViewResNetTeacher(_DualViewBase):
    def __init__(self, num_classes=NUM_CLASSES, feat_dim=2048, fusion_type=FUSION_TYPE, init_thresholds=None):
        super().__init__()
        bb = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2); bb.fc = nn.Identity()
        self.backbone = bb
        self.fusion = InteractionFusion(feat_dim, fusion_type)
        self.main_head   = CORALHead(self.fusion.out_dim, num_classes, init_thresholds)
        self.macula_head = CORALHead(feat_dim, num_classes, init_thresholds)
        self.disc_head   = CORALHead(feat_dim, num_classes, init_thresholds)


class DualViewLightStudent(_DualViewBase):
    def __init__(self, num_classes=NUM_CLASSES, backbone=None, fusion_type=FUSION_TYPE, init_thresholds=None):
        super().__init__()
        self.backbone = backbone or LightweightBackbone()
        fd = self.backbone.out_dim
        self.fusion = InteractionFusion(fd, fusion_type)
        self.main_head   = CORALHead(self.fusion.out_dim, num_classes, init_thresholds)
        self.macula_head = CORALHead(fd, num_classes, init_thresholds)
        self.disc_head   = CORALHead(fd, num_classes, init_thresholds)
    def fuse_model(self, qat=False):
        if hasattr(self.backbone, "fuse_model"): self.backbone.fuse_model(qat=qat)

INIT_THRESHOLDS = compute_init_thresholds(DRTID_TRAIN_CSV)
POS_WEIGHT = compute_pos_weights(DRTID_TRAIN_CSV)
print("CORAL init thresholds :", [round(t, 4) for t in INIT_THRESHOLDS])
print("  implied P(y>k)      :", [round(float(torch.sigmoid(torch.tensor(t))), 4) for t in INIT_THRESHOLDS])
print(f"pos_weight ({POS_WEIGHT_MODE:4s})     :", [round(float(w), 3) for w in POS_WEIGHT])

In [ ]:
# ---- Unit tests on the ordinal head (fail loudly, before any training) ----
def test_coral_head():
    h = CORALHead(16, NUM_CLASSES, INIT_THRESHOLDS)
    b = h._ordered_biases().detach()
    assert torch.all(b[:-1] >= b[1:]), "thresholds must be non-increasing"
    spread = float(b[0] - b[-1])
    assert spread > 1.5, f"threshold spread {spread:.3f} too small -- predictions will collapse to extremes"
    _, p = h(torch.randn(32, 16))
    assert torch.all(p[:, :-1] >= p[:, 1:] - 1e-6), "P(y>k) must be non-increasing in k"
    emp = [float(torch.sigmoid(torch.tensor(t))) for t in INIT_THRESHOLDS]
    got = [float(torch.sigmoid(x)) for x in b]
    assert max(abs(a - c) for a, c in zip(emp, got)) < 1e-5, "init must reproduce empirical marginals"
    print(f"  CORAL unit tests PASSED (spread={spread:.3f} logits, monotone, matches marginals)")
    return spread

_spread = test_coral_head()

def test_fusion_interaction():
    f = InteractionFusion(8, "interaction_mlp").eval()
    a, b = torch.randn(4, 8), torch.randn(4, 8)
    assert not torch.allclose(f(a, b), f(b, a)), "fusion must not be order-invariant (it models interaction)"
    print("  InteractionFusion unit test PASSED (view-order sensitive => genuine interaction)")

test_fusion_interaction()
record_gate("Gate_CORAL_UnitTests", True, f"threshold spread {_spread:.3f} logits; monotone; matches marginals")

## 10 — Loss definitions

`L = L_task + λ·L_aux + α·L_logitKD + β·L_CSD (+ γ·L_featKD)`

**CSD (normalized).** `Δ = p_dual − (p_macula+p_disc)/2` for teacher and student; both are divided
by `s = mean(|Δ^T|)` (detached) before a Huber loss. Because the divisor is detached and identical
on both sides, the optimum is unchanged but the gradient becomes usable: rev2 logged `L_CSD≈0.014`
against `L_task≈0.82` (<0.5% of the objective, essentially no gradient), which is why its RQ1 test
could not have detected any CSD effect.

In [ ]:
def coral_loss(logits, labels, num_thresholds=NUM_THRESHOLDS, pos_weight=None):
    levels = torch.arange(num_thresholds, device=logits.device).unsqueeze(0)
    y_k = (labels.unsqueeze(1) > levels).float()
    return F.binary_cross_entropy_with_logits(logits, y_k, pos_weight=pos_weight)

def aux_loss(out, labels, pos_weight=None):
    return (coral_loss(out["logit_macula"], labels, pos_weight=pos_weight) +
            coral_loss(out["logit_disc"],   labels, pos_weight=pos_weight))

def logit_kd_loss(logit_t, logit_s, tau=2.0):
    """No tau^2 factor (judge.md Flag 17): alpha and tau are therefore coupled -- do not claim
    independent temperature tuning in the paper."""
    return F.binary_cross_entropy(torch.sigmoid(logit_s / tau), torch.sigmoid(logit_t.detach() / tau))

def _delta(p_dual, p_m, p_d):
    return p_dual - (p_m + p_d) / 2

def csd_loss(p_dual_t, p_m_t, p_d_t, p_dual_s, p_m_s, p_d_s,
             variant="smoothl1_norm", tau_csd=0.5, huber_beta=1.0, eps=1e-6):
    dt = _delta(p_dual_t.detach(), p_m_t.detach(), p_d_t.detach())
    ds = _delta(p_dual_s, p_m_s, p_d_s)
    if variant == "smoothl1_norm":                      # DEFAULT
        s = dt.abs().mean().detach().clamp_min(1e-3)
        return F.smooth_l1_loss(ds / s, dt / s, beta=huber_beta)
    if variant == "smoothl1":                           # rev2 formulation (ablation)
        return F.smooth_l1_loss(ds, dt, beta=huber_beta)
    if variant == "magnitude_weighted_direction":       # judge.md Flag 5
        mag = dt.norm(dim=1)
        w = (mag / mag.median().clamp_min(eps)).clamp(max=1.0)   # near-zero teacher shifts carry no direction
        l_dir = ((1 - F.cosine_similarity(ds, dt, dim=1, eps=eps)) * w).sum() / w.sum().clamp_min(eps)
        s = dt.abs().mean().detach().clamp_min(1e-3)
        return 0.5 * l_dir + 0.5 * F.smooth_l1_loss(ds / s, dt / s, beta=huber_beta)
    if variant == "kl_softmax":                         # v1 formulation (negative control)
        return F.kl_div(F.log_softmax(ds / tau_csd, dim=1), F.softmax(dt / tau_csd, dim=1), reduction="batchmean")
    raise ValueError(f"unknown csd_variant: {variant}")

def feature_kd_loss(z_t, z_s, projector):
    """Representation-level control: isolates whether DECISION-shift knowledge is special versus
    ordinary feature transfer."""
    return F.mse_loss(z_s, projector(z_t.detach()))

def get_student_output(student, macula, disc, view_mode):
    if view_mode == "dual":        return student(macula, disc)
    if view_mode == "macula_only": return student.forward_single(macula, "macula")
    if view_mode == "disc_only":   return student.forward_single(disc, "disc")
    raise ValueError(view_mode)

def ordinal_violation_rate(p):
    return float((p[:, 1:] - p[:, :-1] > 0).float().mean())

def combined_student_loss(teacher_out, student_out, labels, view_mode, alpha=0.0, beta=0.0,
                          lambda_aux=0.5, tau_kd=2.0, csd_variant="smoothl1_norm", tau_csd=0.5,
                          pos_weight=None, use_counterfactual_csd=False, teacher_cf_out=None,
                          student_cf_out=None, gamma_feat=0.0, feat_projector=None, huber_beta=1.0):
    task_logit = student_out["logit_dual"] if view_mode == "dual" else student_out["logit"]
    l_task = coral_loss(task_logit, labels, pos_weight=pos_weight)
    total, log, comps = l_task, {"L_task": l_task.item()}, {"task": l_task}

    if view_mode == "dual":
        l_aux = aux_loss(student_out, labels, pos_weight=pos_weight)
        total = total + lambda_aux * l_aux
        log["L_aux"] = l_aux.item(); log["W_aux"] = float(lambda_aux * l_aux)
        comps["aux"] = lambda_aux * l_aux
        if alpha > 0:
            l_kd = logit_kd_loss(teacher_out["logit_dual"], student_out["logit_dual"], tau_kd)
            total = total + alpha * l_kd
            log["L_logit_KD"] = l_kd.item(); log["W_logit_KD"] = float(alpha * l_kd)
            comps["logit_kd"] = alpha * l_kd
        if beta > 0:
            if use_counterfactual_csd:
                l_csd = csd_loss(teacher_cf_out["p_dual"], teacher_cf_out["p_macula_cf"], teacher_cf_out["p_disc_cf"],
                                 student_cf_out["p_dual"], student_cf_out["p_macula_cf"], student_cf_out["p_disc_cf"],
                                 variant=csd_variant, tau_csd=tau_csd, huber_beta=huber_beta)
            else:
                l_csd = csd_loss(teacher_out["p_dual"], teacher_out["p_macula"], teacher_out["p_disc"],
                                 student_out["p_dual"], student_out["p_macula"], student_out["p_disc"],
                                 variant=csd_variant, tau_csd=tau_csd, huber_beta=huber_beta)
            total = total + beta * l_csd
            log["L_CSD"] = l_csd.item(); log["W_CSD"] = float(beta * l_csd)
            comps["csd"] = beta * l_csd
        if gamma_feat > 0 and feat_projector is not None:
            l_f = feature_kd_loss(teacher_out["z_fused"], student_out["z_fused"], feat_projector)
            total = total + gamma_feat * l_f
            log["L_feat_KD"] = l_f.item(); log["W_feat_KD"] = float(gamma_feat * l_f)
            comps["feat_kd"] = gamma_feat * l_f

    log["L_total"] = total.item()
    return total, log, comps

def component_grad_norms(components, params):
    """Per-component gradient norms. Loss VALUES alone cannot establish that a term influences
    learning -- rev2 logged values only, which is why its dead CSD term stayed invisible until the
    logs were re-read by hand after the entire run had finished."""
    out, params = {}, [p for p in params if p.requires_grad]
    for name, t in components.items():
        if t is None or not t.requires_grad: continue
        g = torch.autograd.grad(t, params, retain_graph=True, allow_unused=True)
        out[f"gnorm_{name}"] = sum(float(x.pow(2).sum()) for x in g if x is not None) ** 0.5
    if out.get("gnorm_task", 0) > 0 and "gnorm_csd" in out:
        out["gnorm_ratio_csd_over_task"] = out["gnorm_csd"] / out["gnorm_task"]
    return out

print("Losses defined.")

## 11 — Complete metrics library

**QWK is the single primary metric** (DR grades are ordinal; a 4-grade error is not the same as a
1-grade error). Everything else is reported as secondary/supplementary — comprehensive coverage,
but explicitly not "everything is primary", which would read as metric fishing.

- *Ordinal*: QWK, MAE, Severe-Error Rate, Ordinal Violation Rate
- *Categorical*: Accuracy, Balanced Accuracy, macro/weighted Precision·Recall·F1
- *Per grade*: Precision, Recall (sensitivity), F1, Specificity (one-vs-rest), Support
- *Calibration*: Brier, ECE
- *Mechanism*: ShiftMAE, CosAgree, BenefitCorr, internal & external dual-view gain

In [ ]:
from sklearn.metrics import (cohen_kappa_score, f1_score, precision_score, recall_score,
                             accuracy_score, balanced_accuracy_score, confusion_matrix)

def fast_qwk(y_true, y_pred, K=NUM_CLASSES):
    """Vectorized QWK -- the bootstrap calls this ~10,000x per comparison."""
    y_true = np.asarray(y_true, int); y_pred = np.asarray(y_pred, int)
    O = np.zeros((K, K)); np.add.at(O, (y_true, y_pred), 1)
    w = (np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2 / (K - 1) ** 2
    ht, hp = np.bincount(y_true, minlength=K), np.bincount(y_pred, minlength=K)
    E = np.outer(ht, hp) / max(len(y_true), 1)
    den = (w * E).sum()
    return 1.0 - (w * O).sum() / den if den > 0 else 0.0

def compute_all_metrics(y_true, y_pred, p_cum=None, K=NUM_CLASSES):
    y_true = np.asarray(y_true, int); y_pred = np.asarray(y_pred, int)
    labels = list(range(K))
    m = {
        "QWK": fast_qwk(y_true, y_pred, K),
        "MAE": float(np.mean(np.abs(y_true - y_pred))),
        "SevereErrorRate": float(np.mean(np.abs(y_true - y_pred) >= 2)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "BalancedAccuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }
    for avg in ("macro", "weighted"):
        tag = "Macro" if avg == "macro" else "Weighted"
        m[f"{tag}Precision"] = float(precision_score(y_true, y_pred, average=avg, labels=labels, zero_division=0))
        m[f"{tag}Recall"]    = float(recall_score(y_true, y_pred, average=avg, labels=labels, zero_division=0))
        m[f"{tag}F1"]        = float(f1_score(y_true, y_pred, average=avg, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    prec = precision_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    total = cm.sum()
    for g in labels:
        tp = cm[g, g]; fn = cm[g, :].sum() - tp; fp = cm[:, g].sum() - tp; tn = total - tp - fn - fp
        m[f"Precision_Grade{g}"]   = float(prec[g])
        m[f"Recall_Grade{g}"]      = float(rec[g])          # sensitivity
        m[f"Sensitivity_Grade{g}"] = float(rec[g])          # alias kept for continuity with rev2/rev3
        m[f"F1_Grade{g}"]          = float(f1[g])
        m[f"Specificity_Grade{g}"] = float(tn / (tn + fp)) if (tn + fp) > 0 else float("nan")
        m[f"Support_Grade{g}"]     = int(cm[g, :].sum())
        m[f"Predicted_Grade{g}"]   = int(cm[:, g].sum())
    if p_cum is not None:
        m["OrdinalViolationRate"] = ordinal_violation_rate(p_cum)
        m.update(compute_calibration(p_cum, y_true))
    return m

def compute_calibration(p_cum, y_true, n_bins=10):
    """Pooled over the K-1 cumulative thresholds. CSD is framed as distilling a shift in
    CONFIDENCE, so calibration is load-bearing for the claim (judge.md Flag 4)."""
    p = p_cum.detach().cpu().float()
    y = torch.as_tensor(np.asarray(y_true, int)).long()
    levels = torch.arange(p.shape[1]).unsqueeze(0)
    t = (y.unsqueeze(1) > levels).float()
    pf, tf = p.flatten(), t.flatten()
    brier = float(((pf - tf) ** 2).mean())
    ece, n, edges = 0.0, pf.numel(), torch.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        msk = (pf > lo) & (pf <= hi) if i > 0 else (pf >= lo) & (pf <= hi)
        if msk.sum() == 0: continue
        ece += float(msk.sum()) / n * abs(float(pf[msk].mean()) - float(tf[msk].mean()))
    return {"Brier": brier, "ECE": ece}

def reliability_curve(p_cum, y_true, n_bins=10):
    p = p_cum.detach().cpu().float()
    y = torch.as_tensor(np.asarray(y_true, int)).long()
    levels = torch.arange(p.shape[1]).unsqueeze(0)
    t = (y.unsqueeze(1) > levels).float()
    pf, tf = p.flatten(), t.flatten()
    edges = torch.linspace(0, 1, n_bins + 1); rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        msk = (pf > lo) & (pf <= hi) if i > 0 else (pf >= lo) & (pf <= hi)
        rows.append({"bin_lo": float(lo), "bin_hi": float(hi), "count": int(msk.sum()),
                     "mean_predicted": float(pf[msk].mean()) if msk.sum() else float("nan"),
                     "observed_frequency": float(tf[msk].mean()) if msk.sum() else float("nan")})
    return pd.DataFrame(rows)

def check_prediction_collapse(y_true, y_pred, K=NUM_CLASSES, recall_floor=0.05):
    """rev2's core pathology only became visible through per-grade recall -- Grades 1-3 were
    essentially never predicted while overall QWK still looked plausible. This makes that failure
    mode impossible to miss again."""
    y_true = np.asarray(y_true, int); y_pred = np.asarray(y_pred, int)
    warns = []
    for g in range(K):
        if (y_pred == g).sum() == 0:
            warns.append(f"Grade {g} NEVER predicted")
        support = (y_true == g).sum()
        if support > 0:
            r = (y_pred[y_true == g] == g).mean()
            if r < recall_floor:
                warns.append(f"Grade {g} recall {r:.3f} < {recall_floor}")
    n_distinct = len(np.unique(y_pred))
    if n_distinct <= 2:
        warns.append(f"prediction distribution COLLAPSED to {n_distinct} distinct grade(s)")
    return warns

print("Metrics library defined.")

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device, view_mode="dual", return_patient_ids=False):
    model.eval()
    preds, targets, ps, pids = [], [], [], []
    for batch in loader:
        macula, disc = batch["macula"].to(device), batch["disc"].to(device)
        out = get_student_output(model, macula, disc, view_mode)
        p = out["p_dual" if view_mode == "dual" else "p"]
        preds.extend((p > 0.5).sum(dim=1).cpu().tolist())
        targets.extend(batch["label"].tolist())
        ps.append(p.cpu())
        if return_patient_ids: pids.extend(batch["patient_id"].tolist())
    y_true, y_pred, p_cum = np.array(targets), np.array(preds), torch.cat(ps, 0)
    return (y_true, y_pred, p_cum, np.array(pids)) if return_patient_ids else (y_true, y_pred, p_cum)

@torch.no_grad()
def quick_val_qwk(model, loader, device, view_mode="dual"):
    y, yp, _ = get_predictions(model, loader, device, view_mode)
    return fast_qwk(y, yp)

@torch.no_grad()
def compute_dual_view_gain(model, loader, device):
    """INTERNAL gain: dual head vs this model's OWN auxiliary heads (shared, jointly-trained
    backbone). Must not be conflated with external gain -- judge.md Flag 8."""
    y, pd_, _ = get_predictions(model, loader, device, "dual")
    _, pm, _  = get_predictions(model, loader, device, "macula_only")
    _, pdd, _ = get_predictions(model, loader, device, "disc_only")
    qd, qm, qdd = fast_qwk(y, pd_), fast_qwk(y, pm), fast_qwk(y, pdd)
    return {"QWK_dual": qd, "QWK_aux_macula": qm, "QWK_aux_disc": qdd,
            "DualViewGain_G_internal": qd - max(qm, qdd)}

def compute_external_gain(qwk_dual, qwk_indep_macula, qwk_indep_disc):
    """EXTERNAL gain: vs INDEPENDENTLY trained single-view students."""
    return qwk_dual - max(qwk_indep_macula, qwk_indep_disc)

def _sample_ordinal_nll(p_cum, y, K=NUM_THRESHOLDS):
    levels = torch.arange(K, device=p_cum.device).unsqueeze(0)
    y_k = (y.unsqueeze(1) > levels).float()
    p = p_cum.clamp(1e-6, 1 - 1e-6)
    return -(y_k * torch.log(p) + (1 - y_k) * torch.log(1 - p)).sum(dim=1)

@torch.no_grad()
def compute_shift_fidelity(teacher, student, loader, device):
    """Did CSD transfer the PATTERN, independently of whether QWK moved? (judge.md Flag 10)

    BenefitCorr is the strongest of the three: it correlates the teacher's and student's per-sample
    fusion benefit B_i = NLL(p_agg) - NLL(p_dual). If complementarity really transferred, the
    student should benefit from dual-view on the SAME samples the teacher does."""
    teacher.eval(); student.eval()
    smae, cos, bt, bs = [], [], [], []
    for batch in loader:
        m, d = batch["macula"].to(device), batch["disc"].to(device)
        y = batch["label"].to(device)
        to, so = teacher(m, d), student(m, d)
        dt = _delta(to["p_dual"], to["p_macula"], to["p_disc"])
        ds = _delta(so["p_dual"], so["p_macula"], so["p_disc"])
        smae.append((ds - dt).abs().sum(1).cpu())
        cos.append(F.cosine_similarity(ds, dt, dim=1, eps=1e-6).cpu())
        for out, sink in ((to, bt), (so, bs)):
            p_agg = (out["p_macula"] + out["p_disc"]) / 2
            sink.append((_sample_ordinal_nll(p_agg, y) - _sample_ordinal_nll(out["p_dual"], y)).cpu())
    smae, cos = torch.cat(smae), torch.cat(cos)
    a, b = torch.cat(bt), torch.cat(bs)
    corr = float(((a - a.mean()) * (b - b.mean())).mean() / (a.std() * b.std())) \
           if a.std() > 1e-8 and b.std() > 1e-8 else float("nan")
    return {"ShiftMAE": float(smae.mean()), "CosAgree": float(cos.mean()), "BenefitCorr": corr,
            "TeacherBenefitPositiveFrac": float((a > 0).float().mean()),
            "StudentBenefitPositiveFrac": float((b > 0).float().mean())}

print("Evaluation helpers defined.")

## 12 — Efficiency benchmark suite (standardized protocol)

Latency is always measured on a **CPU copy** of the model — there is no code path that can time a
CUDA model and label it CPU. Protocol is fixed and recorded with every measurement:
`batch=1, warmup=50, runs=500, threads=1`.

Size comparisons use **equivalent deployment artifacts** (FP32 export vs INT8 export), never an
FP32 training checkpoint against an INT8 state_dict.

In [ ]:
try:
    import psutil; _HAS_PSUTIL = True
except Exception:
    _HAS_PSUTIL = False

def param_count(model):
    return int(sum(p.numel() for p in model.parameters()))

def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2) if path and os.path.exists(path) else float("nan")

def benchmark_latency(model, macula, disc, view_mode="dual", bench=BENCH, on_cpu=True, label=""):
    """Returns mean/median/SD/p95/p99/throughput plus the protocol used."""
    torch.set_num_threads(bench["threads"])
    if on_cpu:
        m = copy.deepcopy(model).to("cpu").eval()
        mac, dsc = macula[:bench["batch_size"]].cpu(), disc[:bench["batch_size"]].cpu()
    else:
        m = model.eval(); mac, dsc = macula[:bench["batch_size"]], disc[:bench["batch_size"]]

    if view_mode == "dual":
        fn = lambda: m(mac, dsc)
    else:
        which = "macula" if "macula" in view_mode else "disc"
        img = mac if which == "macula" else dsc
        fn = lambda: m.forward_single(img, which=which)

    with torch.no_grad():
        for _ in range(bench["warmup"]): fn()
        ts = []
        for _ in range(bench["runs"]):
            t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1000)
    ts = np.array(ts)
    med = float(np.median(ts))
    return {"Latency_mean_ms": float(ts.mean()), "Latency_median_ms": med,
            "Latency_sd_ms": float(ts.std()), "Latency_p95_ms": float(np.percentile(ts, 95)),
            "Latency_p99_ms": float(np.percentile(ts, 99)),
            "Throughput_img_per_s": float(1000.0 / med) if med > 0 else float("nan"),
            "bench_device": "cpu" if on_cpu else "cuda", "bench_threads": bench["threads"],
            "bench_warmup": bench["warmup"], "bench_runs": bench["runs"],
            "bench_batch_size": bench["batch_size"]}

def measure_memory(build_fn, macula, disc):
    """Peak resident memory around model construction + one inference (CPU-side, deployment-relevant)."""
    if not _HAS_PSUTIL: return {"PeakRSS_MB": float("nan"), "InferenceRSSDelta_MB": float("nan")}
    proc = psutil.Process(os.getpid())
    base = proc.memory_info().rss / 1024 ** 2
    m = build_fn()
    loaded = proc.memory_info().rss / 1024 ** 2
    with torch.no_grad(): m(macula[:1].cpu(), disc[:1].cpu())
    after = proc.memory_info().rss / 1024 ** 2
    del m
    return {"ModelLoadRSS_MB": max(loaded - base, 0.0), "InferenceRSSDelta_MB": max(after - loaded, 0.0),
            "PeakRSS_MB": after}

def efficiency_derived(size_mb, ref_size_mb, latency_ms, ref_latency_ms):
    out = {}
    if ref_size_mb and size_mb and not math.isnan(size_mb) and not math.isnan(ref_size_mb) and size_mb > 0:
        out["CompressionRatio_vs_ref"] = ref_size_mb / size_mb
        out["SizeReduction_pct"] = (1 - size_mb / ref_size_mb) * 100
    if ref_latency_ms and latency_ms and latency_ms > 0:
        out["Speedup_vs_ref"] = ref_latency_ms / latency_ms
    return out

def retention_metrics(m_comp, m_ref, keys=("QWK", "Accuracy", "MacroF1", "MacroRecall")):
    """Retention for score-type metrics; for error-type metrics report the DELTA instead, since a
    ratio of errors is not interpretable in the same direction."""
    out = {}
    for k in keys:
        if k in m_comp and k in m_ref and m_ref[k] not in (0, None) and not math.isnan(m_ref[k]):
            out[f"{k}_retention_pct"] = 100.0 * m_comp[k] / m_ref[k]
    for k in ("MAE", "SevereErrorRate", "ECE", "Brier"):
        if k in m_comp and k in m_ref:
            out[f"delta_{k}"] = m_comp[k] - m_ref[k]
    return out

CPU_INFO = platform.processor() or "unknown"
try:
    CPU_INFO = subprocess.check_output("lscpu | grep 'Model name' | head -1", shell=True, text=True).split(":")[-1].strip() or CPU_INFO
except Exception:
    pass
print("Benchmark CPU:", CPU_INFO, "| protocol:", BENCH)

## 13 — FP32 smoke tests (must pass before any training)

In [ ]:
from tqdm.auto import tqdm

def smoke_test():
    ds = DRTiDDualViewDataset(DRTID_TRAIN_CSV, eval_transform)
    batch = next(iter(make_loader(ds, 8, True, workers=0)))
    m, d, y = batch["macula"].to(DEVICE), batch["disc"].to(DEVICE), batch["label"].to(DEVICE)

    teacher = DualViewResNetTeacher(init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    student = DualViewLightStudent(init_thresholds=INIT_THRESHOLDS).to(DEVICE)

    n_s, n_sb, n_t = param_count(student), param_count(student.backbone), param_count(teacher)
    print(f"Student {n_s:,} params (backbone {n_sb:,}) | Teacher {n_t:,} | compression {n_t/n_s:.0f}x")
    assert n_s > 150_000, f"student too small ({n_s:,}) -- rev2's 34K student was capacity-capped"

    t_out = teacher(m, d)
    ovr = ordinal_violation_rate(t_out["p_dual"])
    assert ovr == 0.0, f"CORAL monotonicity broken: OVR={ovr}"

    _ = teacher.forward_single(m, "macula")
    cf = teacher.counterfactual_forward(m, d)

    for vm in ["dual", "macula_only", "disc_only"]:
        s_out = get_student_output(student, m, d, vm)
        loss, log, comps = combined_student_loss(t_out, s_out, y, vm, alpha=0.5, beta=1.0,
                                                 pos_weight=POS_WEIGHT.to(DEVICE))
        loss.backward(); student.zero_grad()
        print(f"  [{vm}] loss={loss.item():.4f}")

    s_cf, s_dual = student.counterfactual_forward(m, d), student(m, d)
    l_cf, _, _ = combined_student_loss(t_out, s_dual, y, "dual", alpha=0.5, beta=1.0,
                                       use_counterfactual_csd=True, teacher_cf_out=cf,
                                       student_cf_out=s_cf, pos_weight=POS_WEIGHT.to(DEVICE))
    l_cf.backward(); student.zero_grad()
    print(f"  [dual + counterfactual CSD] loss={l_cf.item():.4f}")

    # Gate 4 precursor: CSD must produce a real gradient, not decoration.
    s_dual = student(m, d)
    _, _, comps = combined_student_loss(t_out, s_dual, y, "dual", alpha=0.5, beta=1.0,
                                        pos_weight=POS_WEIGHT.to(DEVICE))
    gn = component_grad_norms(comps, list(student.fusion.parameters()) + list(student.main_head.parameters()))
    student.zero_grad()
    ratio = gn.get("gnorm_ratio_csd_over_task", 0.0)
    print("  gradient norms:", {k: round(v, 4) for k, v in gn.items()})
    assert ratio > 0.01, f"CSD gradient ratio {ratio:.5f} negligible -- rev2's failure mode"
    print(f"  CSD/task gradient ratio at beta=1.0 (probe value): {ratio:.3f}")
    if ratio > 3.0:
        print("  NOTE: at beta=1.0 CSD would dominate the task loss. That is the OPPOSITE of rev2's")
        print("        failure and equally harmful, which is exactly why the Section 22 grid searches")
        print("        beta in [0.1, 0.5] and lets validation choose rather than fixing beta by hand.")

    student.eval(); student.fuse_model()
    print("  fuse_model() OK")
    print("\nSMOKE TEST PASSED.")
    return {"student_params": n_s, "teacher_params": n_t, "csd_grad_ratio": ratio}

SMOKE = smoke_test()
record_gate("Gate_SmokeTest_FP32", True,
            f"student={SMOKE['student_params']:,} teacher={SMOKE['teacher_params']:,} "
            f"csd/task grad ratio={SMOKE['csd_grad_ratio']:.3f}")

## 14 — APTOS backbone pretraining

In [ ]:
def build_backbone(kind):
    if kind == "resnet50":
        m = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Identity(); return m, 2048
    if kind == "lightweight":
        m = LightweightBackbone(); return m, m.out_dim
    raise ValueError(kind)

def pretrain_backbone(kind, epochs, lr, batch_size, seed=PRIMARY_SEED, force=False):
    out = f"{CKPT_DIR}/pretrained_backbones/aptos_{kind}_backbone.pt"
    backbone, feat_dim = build_backbone(kind)
    if not force and checkpoint_is_compatible(out, backbone, unwrap_key=None):
        print(f"{kind}: compatible checkpoint exists, skipping."); return out
    set_seed(seed)
    pw = compute_pos_weights(f"{APTOS_ROOT}/train_1.csv", grade_col="diagnosis").to(DEVICE)
    tl = make_loader(APTOSSingleViewDataset(f"{APTOS_ROOT}/train_1.csv", f"{APTOS_ROOT}/train_images/train_images",
                                            aptos_train_transform), batch_size, True, seed)
    vl = make_loader(APTOSSingleViewDataset(f"{APTOS_ROOT}/valid.csv", f"{APTOS_ROOT}/val_images/val_images",
                                            aptos_eval_transform), batch_size, False)
    head = CORALHead(feat_dim, NUM_CLASSES, INIT_THRESHOLDS)
    backbone, head = backbone.to(DEVICE), head.to(DEVICE)
    opt = torch.optim.AdamW(list(backbone.parameters()) + list(head.parameters()), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.02)

    best, hist = -1.0, []
    for ep in range(epochs):
        backbone.train(); head.train()
        for b in tqdm(tl, desc=f"[pretrain-{kind}] ep{ep}", leave=False):
            img, y = b["image"].to(DEVICE), b["label"].to(DEVICE)
            loss = coral_loss(head(backbone(img))[0], y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        backbone.eval(); head.eval()
        pr, tg = [], []
        with torch.no_grad():
            for b in vl:
                _, p = head(backbone(b["image"].to(DEVICE)))
                pr.extend((p > 0.5).sum(1).cpu().tolist()); tg.extend(b["label"].tolist())
        q = fast_qwk(tg, pr); hist.append({"epoch": ep, "val_qwk": q})
        print(f"  ep{ep}: val_QWK={q:.4f}")
        if q > best:
            best = q; robust_torch_save(backbone.state_dict(), out)
    pd.DataFrame(hist).to_csv(f"{LOGS_DIR}/pretrain_{kind}_history.csv", index=False)
    assert best > 0.0, f"pretrain {kind} QWK<=0 -- worse than majority baseline"
    print(f"{kind} pretrain done. best val QWK={best:.4f}")
    return out

RESNET50_BACKBONE_CKPT   = pretrain_backbone("resnet50",   epochs=20, lr=1e-4, batch_size=32)
LIGHTWEIGHT_BACKBONE_CKPT = pretrain_backbone("lightweight", epochs=30, lr=1e-3, batch_size=32)

## 15 — Teacher training & Gate 2

In [ ]:
def train_teacher(freeze_epochs=5, finetune_epochs=20, patience=8, lambda_aux=0.3,
                  seed=PRIMARY_SEED, batch_size=16, freeze_lr=3e-4, finetune_lr=1e-5, force=False):
    out = f"{CKPT_DIR}/teacher/teacher_final.pt"
    model = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out, model, "model_state"):
        print("Teacher checkpoint compatible, skipping."); return out
    model = model.to(DEVICE)
    set_seed(seed)
    pw = POS_WEIGHT.to(DEVICE)
    tl = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, train_transform), batch_size, True, seed)
    vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size, False)
    model.backbone.load_state_dict(robust_torch_load(RESNET50_BACKBONE_CKPT, map_location=DEVICE))

    best, hist, holder = -1.0, [], {"state": None}
    def run(epochs, lr, best):
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=lr * 0.02)
        bad = 0
        for _ in range(epochs):
            ge = len(hist); model.train()
            for b in tqdm(tl, desc=f"[teacher] ep{ge}", leave=False):
                m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
                o = model(m, d)
                loss = coral_loss(o["logit_dual"], y, pos_weight=pw) + lambda_aux * aux_loss(o, y, pos_weight=pw)
                opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
            q = quick_val_qwk(model, vl, DEVICE, "dual")
            hist.append({"epoch": ge, "val_qwk": q, "lr": sch.get_last_lr()[0]})
            print(f"  ep{ge}: val_QWK={q:.4f}")
            if q > best:
                best, bad = q, 0
                holder["state"] = copy.deepcopy(model.state_dict())
                robust_torch_save({"model_state": holder["state"], "epoch": ge, "val_qwk": q}, out)
            else:
                bad += 1
                if bad >= patience: print(f"  early stop @ep{ge}"); break
        return best

    for p in model.backbone.parameters(): p.requires_grad = False
    best = run(freeze_epochs, freeze_lr, best)
    # Reload best freeze-phase weights from MEMORY, not Drive -- a write-then-immediate-read of the
    # same path can hit Drive's FUSE sync lag and raise FileNotFoundError mid-run.
    if holder["state"] is not None: model.load_state_dict(holder["state"])
    for p in model.backbone.parameters(): p.requires_grad = True
    best = run(finetune_epochs, finetune_lr, best)

    pd.DataFrame(hist).to_csv(f"{LOGS_DIR}/teacher_history.csv", index=False)
    print(f"Teacher done. best val QWK={best:.4f}")
    return out

TEACHER_CKPT = train_teacher()

_t = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
_t.load_state_dict(robust_torch_load(TEACHER_CKPT, map_location=DEVICE)["model_state"])
_vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), 16, False)
_gain = compute_dual_view_gain(_t, _vl, DEVICE)
GATE2_PASSED = _gain["QWK_dual"] > max(_gain["QWK_aux_macula"], _gain["QWK_aux_disc"])
print("Gate 2 (validation):", {k: round(v, 4) for k, v in _gain.items()})
record_gate("Gate2_Teacher", GATE2_PASSED,
            f"QWK_dual={_gain['QWK_dual']:.4f} vs max(aux)={max(_gain['QWK_aux_macula'], _gain['QWK_aux_disc']):.4f} "
            f"G_internal={_gain['DualViewGain_G_internal']:+.4f}")
if not GATE2_PASSED:
    print("*** Teacher shows no dual-view advantage. CSD's Delta is only meaningful if it does. ***")
    print("*** Levers: lambda_aux lower, freeze_lr lower, more epochs, or a different seed.      ***")
del _t, _vl

## 16 — Generic student trainer

One function drives every student condition, so training logic cannot drift between conditions.
Logs per-epoch loss values, **weighted contributions**, and **per-component gradient norms** to
`gradient_contributions_<condition>_<seed>.csv`.

In [ ]:
_teacher_cache = None
def get_teacher():
    global _teacher_cache
    if _teacher_cache is None:
        t = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
        t.load_state_dict(robust_torch_load(TEACHER_CKPT, map_location=DEVICE)["model_state"])
        t.eval()
        for p in t.parameters(): p.requires_grad = False
        _teacher_cache = t
    return _teacher_cache

def train_student_condition(run_name, seed, view_mode, alpha=0.0, beta=0.0, lambda_aux=0.5,
                            csd_variant="smoothl1_norm", tau_kd=2.0, tau_csd=0.5,
                            use_counterfactual_csd=False, epochs=40, patience=8, lr=1e-3,
                            batch_size=16, gamma_feat=0.0, huber_beta=1.0, weight_decay=1e-4,
                            force=False):
    ck_dir = f"{CKPT_DIR}/student/{run_name}"; os.makedirs(ck_dir, exist_ok=True)
    out = f"{ck_dir}/best_seed{seed}.pt"
    probe = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out, probe, "model_state"):
        print(f"{run_name}|seed{seed}: compatible checkpoint exists, skipping.")
        return out, None
    del probe

    cfg = dict(run_name=run_name, seed=seed, view_mode=view_mode, alpha=alpha, beta=beta,
               lambda_aux=lambda_aux, csd_variant=csd_variant, tau_kd=tau_kd, tau_csd=tau_csd,
               use_counterfactual_csd=use_counterfactual_csd, epochs=epochs, patience=patience,
               lr=lr, batch_size=batch_size, gamma_feat=gamma_feat, huber_beta=huber_beta,
               weight_decay=weight_decay, pos_weight_mode=POS_WEIGHT_MODE)
    save_json(cfg, f"{CONFIG_DIR}/{run_name}_seed{seed}.json")

    set_seed(seed)
    pw = POS_WEIGHT.to(DEVICE)
    tl = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, train_transform), batch_size, True, seed)
    vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size, False)

    teacher = get_teacher()
    student = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    student.backbone.load_state_dict(robust_torch_load(LIGHTWEIGHT_BACKBONE_CKPT, map_location=DEVICE))

    feat_proj, trainable = None, list(student.parameters())
    if gamma_feat > 0:
        feat_proj = nn.Linear(teacher.fusion.out_dim, student.fusion.out_dim).to(DEVICE)
        trainable += list(feat_proj.parameters())

    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.02)
    probe_params = list(student.fusion.parameters()) + list(student.main_head.parameters())
    best, bad, hist = -1.0, 0, []

    for ep in range(epochs):
        student.train(); logs, gnorms = [], {}
        for bi, b in enumerate(tqdm(tl, desc=f"[{run_name}|s{seed}] ep{ep}", leave=False)):
            m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
            with torch.no_grad():
                t_out = teacher(m, d)
                t_cf = teacher.counterfactual_forward(m, d) if use_counterfactual_csd else None
            s_out = get_student_output(student, m, d, view_mode)
            s_cf = student.counterfactual_forward(m, d) if (use_counterfactual_csd and view_mode == "dual") else None

            loss, log, comps = combined_student_loss(
                t_out, s_out, y, view_mode, alpha=alpha, beta=beta, lambda_aux=lambda_aux,
                tau_kd=tau_kd, csd_variant=csd_variant, tau_csd=tau_csd, pos_weight=pw,
                use_counterfactual_csd=use_counterfactual_csd, teacher_cf_out=t_cf,
                student_cf_out=s_cf, gamma_feat=gamma_feat, feat_projector=feat_proj,
                huber_beta=huber_beta)
            if bi == 0 and view_mode == "dual":
                gnorms = component_grad_norms(comps, probe_params); student.zero_grad(set_to_none=True)
            opt.zero_grad(); loss.backward(); opt.step()
            logs.append(log)
        sch.step()

        q = quick_val_qwk(student, vl, DEVICE, view_mode)
        row = {k: float(np.mean([l[k] for l in logs if k in l])) for k in logs[0]}
        row.update({"epoch": ep, "val_qwk": q, "lr": sch.get_last_lr()[0], **gnorms})
        hist.append(row)
        ls = {k: round(v, 4) for k, v in row.items() if k.startswith("L_")}
        gs = {k.replace("gnorm_", ""): round(v, 3) for k, v in gnorms.items()}
        print(f"  ep{ep}: val_QWK={q:.4f} losses={ls}" + (f" grad={gs}" if gs else ""))

        if q > best:
            best, bad = q, 0
            robust_torch_save({"model_state": student.state_dict(), "epoch": ep, "val_qwk": q,
                               "seed": seed, "config": cfg}, out)
        else:
            bad += 1
            if bad >= patience: print(f"  early stop @ep{ep}"); break

    hdf = pd.DataFrame(hist)
    hdf.to_csv(f"{LOGS_DIR}/{run_name}_seed{seed}_history.csv", index=False)
    gcols = [c for c in hdf.columns if c.startswith("gnorm_") or c.startswith("W_") or c.startswith("L_")]
    if gcols:
        hdf[["epoch"] + gcols].to_csv(f"{LOGS_DIR}/gradient_contributions_{run_name}_{seed}.csv", index=False)
    print(f"[{run_name}|s{seed}] best val QWK={best:.4f}")
    return out, hist

print("Student trainer defined.")

## 17–20 — Baselines: single-view, no-distill, logit-KD, feature-KD

In [ ]:
# Single-view baselines (3 seeds). These double as the reference for EXTERNAL dual-view gain,
# since they are the only models that never see two views.
for s in SEEDS_BASELINE:
    train_student_condition("macula_only", seed=s, view_mode="macula_only")
    train_student_condition("disc_only",   seed=s, view_mode="disc_only")

In [ ]:
for s in SEEDS_CORE:
    train_student_condition("dual_no_distill", seed=s, view_mode="dual", alpha=0.0, beta=0.0, lambda_aux=0.5)

In [ ]:
for s in SEEDS_CORE:
    train_student_condition("dual_logitkd", seed=s, view_mode="dual", alpha=0.5, beta=0.0,
                            lambda_aux=0.5, tau_kd=2.0)

In [ ]:
# Representation-level control: isolates whether decision-shift knowledge is special versus
# ordinary feature transfer. Without this, "CSD beats no-distillation" would be a much weaker claim.
for s in SEEDS_CORE:
    train_student_condition("dual_featkd", seed=s, view_mode="dual", alpha=0.5, beta=0.0,
                            lambda_aux=0.5, tau_kd=2.0, gamma_feat=1.0)

## 21–23 — Gate 4 (CSD signal), grid search, final CSD training

In [ ]:
@torch.no_grad()
def gate4_csd_signal(teacher, loader, device):
    """Is there a non-trivial complementarity shift to distil at all?"""
    teacher.eval(); norms = []
    for b in loader:
        o = teacher(b["macula"].to(device), b["disc"].to(device))
        norms.append(_delta(o["p_dual"], o["p_macula"], o["p_disc"]).abs().sum(1).cpu())
    n = torch.cat(norms)
    stats = {"mean_L1": float(n.mean()), "median_L1": float(n.median()),
             "q25": float(n.quantile(.25)), "q75": float(n.quantile(.75)),
             "frac_gt_0.02": float((n > 0.02).float().mean())}
    pd.DataFrame({"delta_L1_norm": n.numpy()}).to_csv(f"{METRICS_DIR}/gate4_teacher_delta_distribution.csv", index=False)
    ok = stats["mean_L1"] >= 1e-3
    record_gate("Gate4_CSD_Signal", ok, ", ".join(f"{k}={v:.4f}" for k, v in stats.items()))
    return stats

_vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), 16, False)
DELTA_STATS = gate4_csd_signal(get_teacher(), _vl, DEVICE)

In [ ]:
# Fixed search space, declared before any result is seen (judge.md Flag 13: validation overfitting).
# Beta range is set from the MEASURED gradient ratio: with the normalized variant, beta=1.0 puts the
# CSD gradient at roughly 5x the task gradient, so the grid brackets clearly-subordinate through
# CSD-leaning rather than guessing. rev2's grid ({0.5,0.7} on the un-normalized loss) could never
# have found a working setting because every point produced <1% of the objective.
GRID = [
    {"csd_variant": "smoothl1_norm", "alpha": 0.5,  "beta": 0.1},
    {"csd_variant": "smoothl1_norm", "alpha": 0.5,  "beta": 0.2},
    {"csd_variant": "smoothl1_norm", "alpha": 0.5,  "beta": 0.5},
    {"csd_variant": "smoothl1_norm", "alpha": 0.25, "beta": 0.2},
    {"csd_variant": "magnitude_weighted_direction", "alpha": 0.5, "beta": 0.2},
]
save_json(GRID, f"{CONFIG_DIR}/csd_grid_space.json")

rows = []
for combo in GRID:
    name = "grid_{}_a{}_b{}".format(combo["csd_variant"], combo["alpha"], combo["beta"])
    ck, hist = train_student_condition(name, seed=PRIMARY_SEED, view_mode="dual", lambda_aux=0.5, **combo)
    st = robust_torch_load(ck, map_location=DEVICE)
    mdl = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    mdl.load_state_dict(st["model_state"])
    y, yp, pc = get_predictions(mdl, _vl, DEVICE, "dual")
    mm = compute_all_metrics(y, yp, pc)
    rows.append({**combo, "run_name": name, "val_QWK": st["val_qwk"],
                 "val_MacroF1": mm["MacroF1"], "val_SevereErrorRate": mm["SevereErrorRate"],
                 "val_MAE": mm["MAE"]})
    del mdl

grid_df = pd.DataFrame(rows).sort_values(["val_QWK", "val_MacroF1"], ascending=[False, False]).reset_index(drop=True)
grid_df.to_csv(f"{TABLES_DIR}/table_01_csd_grid_search.csv", index=False)
print(grid_df.to_string(index=False))

BEST_CSD_VARIANT = grid_df.iloc[0]["csd_variant"]
BEST_ALPHA       = float(grid_df.iloc[0]["alpha"])
BEST_BETA        = float(grid_df.iloc[0]["beta"])
print(f"\nSelected on VALIDATION only -> variant={BEST_CSD_VARIANT} alpha={BEST_ALPHA} beta={BEST_BETA}")
save_json({"csd_variant": BEST_CSD_VARIANT, "alpha": BEST_ALPHA, "beta": BEST_BETA},
          f"{CONFIG_DIR}/csd_selected.json")

In [ ]:
for s in SEEDS_CORE:
    train_student_condition("dual_csd", seed=s, view_mode="dual", alpha=BEST_ALPHA, beta=BEST_BETA,
                            lambda_aux=0.5, csd_variant=BEST_CSD_VARIANT, tau_kd=2.0, tau_csd=0.5)

In [ ]:
# ---- CSD ablations (spec 13): formulation controls, at the selected alpha/beta ----
for s in SEEDS_BASELINE:
    train_student_condition("abl_csd_raw_smoothl1", seed=s, view_mode="dual", alpha=BEST_ALPHA,
                            beta=0.7, lambda_aux=0.5, csd_variant="smoothl1")          # rev2 formulation
for s in SEEDS_BASELINE:
    train_student_condition("abl_csd_kl_softmax", seed=s, view_mode="dual", alpha=BEST_ALPHA,
                            beta=0.5, lambda_aux=0.5, csd_variant="kl_softmax")        # v1 negative control
train_student_condition("abl_csd_counterfactual", seed=PRIMARY_SEED, view_mode="dual",
                        alpha=BEST_ALPHA, beta=BEST_BETA, lambda_aux=0.5,
                        csd_variant=BEST_CSD_VARIANT, use_counterfactual_csd=True)     # judge.md Flag 1/3

## 24–25 — Model selection (validation only) & best FP32 deployment candidate

`M* = argmax QWK_val` over {no-distill, logit-KD, feature-KD, CSD}. Ties within 0.005 resolve by
Macro-F1 → lower severe-error → lower MAE → simpler method. **The test set is not consulted.**

In [ ]:
CORE_CONDITIONS = ["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd"]

def collect_val_scores():
    rows = []
    for cond in CORE_CONDITIONS:
        for s in SEEDS_CORE:
            ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
            if not os.path.exists(ck): continue
            st = robust_torch_load(ck, map_location="cpu")
            mdl = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
            mdl.load_state_dict(st["model_state"])
            y, yp, pc = get_predictions(mdl, _vl, DEVICE, "dual")
            m = compute_all_metrics(y, yp, pc)
            rows.append({"condition": cond, "seed": s, "checkpoint": ck, **{k: m[k] for k in
                         ("QWK", "MacroF1", "SevereErrorRate", "MAE", "Accuracy")}})
            del mdl
    return pd.DataFrame(rows)

VAL_SCORES = collect_val_scores()
VAL_SCORES.to_csv(f"{TABLES_DIR}/table_02_validation_scores.csv", index=False)
print(VAL_SCORES.groupby("condition")[["QWK", "MacroF1", "SevereErrorRate", "MAE"]].agg(["mean", "std"]).round(4).to_string())

def select_best(df):
    d = df.sort_values("QWK", ascending=False).reset_index(drop=True)
    top = d.iloc[0]["QWK"]
    tied = d[d["QWK"] >= top - SELECTION_TIE_EPS]
    if len(tied) > 1:
        print(f"  {len(tied)} candidates within {SELECTION_TIE_EPS} QWK -- applying tie-break chain")
        tied = tied.sort_values(["MacroF1", "SevereErrorRate", "MAE"], ascending=[False, True, True])
    return tied.iloc[0]

BEST_ROW = select_best(VAL_SCORES)
BEST_CONDITION, BEST_SEED = BEST_ROW["condition"], int(BEST_ROW["seed"])
BEST_FP32_CKPT = BEST_ROW["checkpoint"]
print(f"\nM* (validation-selected) = {BEST_CONDITION} seed {BEST_SEED} | val QWK={BEST_ROW['QWK']:.4f}")

# The best CSD model is tracked separately: even if M* is not CSD, the paper still needs the best
# CSD artifact for the RQ1 mechanism analysis.
_csd = VAL_SCORES[VAL_SCORES.condition == "dual_csd"]
BEST_CSD_SEED = int(select_best(_csd)["seed"]) if len(_csd) else PRIMARY_SEED
BEST_CSD_CKPT = f"{CKPT_DIR}/student/dual_csd/best_seed{BEST_CSD_SEED}.pt"
print(f"Best CSD artifact        = dual_csd seed {BEST_CSD_SEED}")
save_json({"best_condition": BEST_CONDITION, "best_seed": BEST_SEED, "best_ckpt": BEST_FP32_CKPT,
           "best_val_qwk": float(BEST_ROW["QWK"]), "best_csd_seed": BEST_CSD_SEED,
           "selection_rule": f"argmax {SELECTION_METRIC}_val, tie<{SELECTION_TIE_EPS} -> {SELECTION_TIEBREAK}"},
          f"{CONFIG_DIR}/model_selection.json")

def load_student(ckpt):
    m = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    m.load_state_dict(robust_torch_load(ckpt, map_location=DEVICE)["model_state"])
    return m.eval()

## 26–29 — Quantization: PTQ INT8, QAT INT8, integrity gates

Modern path first (`torch.export` + PT2E/torchao), eager mode as fallback. **The path actually used
is reported explicitly** — a fallback is never silently presented as the modern path having worked.

Only the CNN backbone is quantized: `torch.cat`, LayerNorm and CORAL's cumsum/softplus/sigmoid have
no eager-mode quantized kernels, and wrapping the whole model fails at runtime the moment a
quantized tensor reaches the fusion block.

In [ ]:
from torch.ao.quantization import (prepare, convert, prepare_qat, get_default_qconfig,
                                   get_default_qat_qconfig, QuantStub, DeQuantStub)

QUANT_ENGINE = "fbgemm" if "fbgemm" in torch.backends.quantized.supported_engines \
               else torch.backends.quantized.supported_engines[0]
torch.backends.quantized.engine = QUANT_ENGINE
print("quantization engine:", QUANT_ENGINE)

class QuantizableBackbone(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.quant = QuantStub(); self.backbone = backbone; self.dequant = DeQuantStub()
    def forward(self, x):
        return self.dequant(self.backbone(self.quant(x)))

def is_quantized_tree(model):
    """Fused quantized modules live under torch.ao.nn.intrinsic.quantized.* -- the class NAME
    (e.g. ConvReLU2d) may not contain 'Quantized', so check the module PATH."""
    return any("quantized" in type(m).__module__.lower() for m in model.modules())

def count_quantized_modules(model):
    return sum(1 for m in model.modules() if "quantized" in type(m).__module__.lower())

def make_calibration_loader(n_batches=64, batch_size=8):
    return make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, eval_transform), batch_size, True, workers=0)

def run_ptq(fp32_ckpt, calib_batches=64):
    """Static PTQ on the backbone. Returns (quantized_model_cpu, info dict)."""
    model = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    model.load_state_dict(robust_torch_load(fp32_ckpt, map_location="cpu")["model_state"])
    model = model.to("cpu").eval()
    model.backbone.fuse_model(qat=False)
    model.backbone = QuantizableBackbone(model.backbone)
    model.backbone.qconfig = get_default_qconfig(QUANT_ENGINE)
    prepared = prepare(model, inplace=False)
    cl = make_calibration_loader(calib_batches)
    with torch.no_grad():
        for i, b in enumerate(cl):
            prepared(b["macula"], b["disc"])
            if i + 1 >= calib_batches: break
    q = convert(prepared, inplace=False)
    return q, {"path": "eager_static_ptq", "engine": QUANT_ENGINE,
               "calib_batches": min(calib_batches, i + 1),
               "quantized_modules": count_quantized_modules(q)}

PTQ_MODEL, PTQ_INFO = None, {}
try:
    PTQ_MODEL, PTQ_INFO = run_ptq(BEST_FP32_CKPT)
    ok = is_quantized_tree(PTQ_MODEL)
    with torch.no_grad():
        _b = next(iter(make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), 4, False, workers=0)))
        _o = PTQ_MODEL(_b["macula"], _b["disc"])
    assert _o["p_dual"].shape[1] == NUM_THRESHOLDS
    record_gate("Gate6_PTQ_Integrity", ok,
                f"{PTQ_INFO['quantized_modules']} quantized modules via {PTQ_INFO['path']} ({QUANT_ENGINE})")
    PTQ_OK = ok
except Exception as e:
    print(f"PTQ FAILED: {e!r}")
    record_gate("Gate6_PTQ_Integrity", False, f"FAILED: {e!r}")
    PTQ_OK = False

In [ ]:
def _build_qat_prepared(fp32_ckpt, load_fp32=True):
    """Deterministic construction of the QAT-prepared model: fuse (train mode) -> wrap -> prepare_qat.
    Factored out so the resume path can rebuild the exact same structure before loading weights."""
    model = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if load_fp32:
        model.load_state_dict(robust_torch_load(fp32_ckpt, map_location="cpu")["model_state"])
    model = model.to("cpu").train()
    model.backbone.fuse_model(qat=True)                    # QAT fusion must run in train mode
    model.backbone = QuantizableBackbone(model.backbone)
    model.backbone.qconfig = get_default_qat_qconfig(QUANT_ENGINE)
    prepare_qat(model, inplace=True)
    return model

def run_qat(fp32_ckpt, epochs=10, lr=3e-5, batch_size=16, patience=4, seed=PRIMARY_SEED, force=False):
    """QAT: start from the best FP32 weights, enable fake quantization, fine-tune on TRAIN,
    select on VALIDATION, then convert. Test set is never involved.

    Resumable: the fine-tuned (pre-conversion) QAT weights are checkpointed to Drive, so a re-run
    after a Colab disconnect reloads them and converts, instead of repeating the fine-tuning epochs.
    Conversion itself is deterministic and cheap, so only the training is worth caching."""
    qat_ckpt = f"{CKPT_DIR}/student/qat/qat_prepared_seed{seed}.pt"
    os.makedirs(os.path.dirname(qat_ckpt), exist_ok=True)

    if not force and os.path.exists(qat_ckpt):
        try:
            saved = robust_torch_load(qat_ckpt, map_location="cpu")
            model = _build_qat_prepared(fp32_ckpt, load_fp32=False)
            model.load_state_dict(saved["model_state"])
            model = model.to("cpu").eval()
            q_model = convert(model, inplace=False)
            info = dict(saved.get("info", {}))
            info["resumed_from_checkpoint"] = True
            info["quantized_modules"] = count_quantized_modules(q_model)
            print(f"QAT: reused fine-tuned checkpoint (best val QWK={info.get('best_val_qwk', float('nan')):.4f}) "
                  f"-- skipped {saved.get('epochs_run', '?')} fine-tuning epochs.")
            return q_model, info
        except Exception as e:
            print(f"QAT checkpoint exists but could not be reused ({e!r}) -- re-running fine-tuning.")

    set_seed(seed)
    model = _build_qat_prepared(fp32_ckpt, load_fp32=True).to(DEVICE)

    pw = POS_WEIGHT.to(DEVICE)
    tl = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, train_transform), batch_size, True, seed)
    vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size, False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    best, bad, best_state, hist = -1.0, 0, None, []
    for ep in range(epochs):
        model.train()
        for b in tqdm(tl, desc=f"[QAT] ep{ep}", leave=False):
            m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
            o = model(m, d)
            loss = coral_loss(o["logit_dual"], y, pos_weight=pw) + 0.5 * aux_loss(o, y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
        q = quick_val_qwk(model, vl, DEVICE, "dual")
        hist.append({"epoch": ep, "val_qwk": q}); print(f"  QAT ep{ep}: val_QWK={q:.4f}")
        if q > best:
            best, bad = q, 0; best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience: print(f"  early stop @ep{ep}"); break

    if best_state is not None: model.load_state_dict(best_state)
    pd.DataFrame(hist).to_csv(f"{LOGS_DIR}/qat_history_seed{seed}.csv", index=False)

    info = {"path": "eager_qat", "engine": QUANT_ENGINE, "best_val_qwk": best,
            "epochs_run": len(hist), "lr": lr, "seed": seed, "resumed_from_checkpoint": False}
    # Checkpoint the fine-tuned PRE-conversion weights so a re-run can skip these epochs.
    robust_torch_save({"model_state": {k: v.cpu() for k, v in model.state_dict().items()},
                       "info": info, "epochs_run": len(hist)}, qat_ckpt)

    model = model.to("cpu").eval()
    q_model = convert(model, inplace=False)
    info["quantized_modules"] = count_quantized_modules(q_model)
    return q_model, info

QAT_MODEL, QAT_INFO, QAT_OK = None, {}, False
try:
    QAT_MODEL, QAT_INFO = run_qat(BEST_FP32_CKPT)
    fake_q_used = QAT_INFO["quantized_modules"] > 0
    with torch.no_grad():
        _b = next(iter(make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), 4, False, workers=0)))
        _o = QAT_MODEL(_b["macula"], _b["disc"])
    QAT_OK = fake_q_used and is_quantized_tree(QAT_MODEL)
    record_gate("Gate7_QAT_Integrity", QAT_OK,
                f"{QAT_INFO['quantized_modules']} quantized modules; QAT best val QWK={QAT_INFO['best_val_qwk']:.4f}")
except Exception as e:
    print(f"QAT FAILED: {e!r}")
    record_gate("Gate7_QAT_Integrity", False, f"FAILED: {e!r}")

save_json({"ptq": PTQ_INFO, "qat": QAT_INFO, "engine": QUANT_ENGINE},
          f"{CONFIG_DIR}/quantization_info.json")

## 30–33 — Full internal test evaluation (Set C touched once)

Every condition × seed is evaluated on the held-out DRTiD test set. **Per-sample predictions are
written to `predictions/<condition>_<seed>.csv`** so any future metric can be recomputed without
re-running inference or touching the models again.

In [ ]:
TEST_DS = DRTiDDualViewDataset(DRTID_TEST_CSV, eval_transform)
TEST_LOADER = make_loader(TEST_DS, 16, False)
_sample = next(iter(TEST_LOADER))
TEACHER = get_teacher()

def save_predictions(condition, seed, y_true, y_pred, p_cum, patient_ids, quantization="FP32",
                     dataset="DRTiD", split="test", latency_ms=np.nan):
    df = pd.DataFrame({"dataset": dataset, "split": split, "patient_id": patient_ids,
                       "sample_id": [f"{dataset}_{p}" for p in patient_ids],
                       "true_grade": y_true, "pred_grade": y_pred,
                       "condition": condition, "seed": seed, "quantization": quantization,
                       "latency_ms": latency_ms})
    for k in range(p_cum.shape[1]):
        df[f"p_threshold_{k}"] = p_cum[:, k].cpu().numpy()
    path = f"{PREDS_DIR}/{dataset}_{split}_{condition}_seed{seed}_{quantization}.csv"
    df.to_csv(path, index=False)
    return path

def save_confusion(condition, seed, y_true, y_pred, tag=""):
    labels = list(range(NUM_CLASSES))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cmn = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1)
    idx = [f"true_{g}" for g in labels]; col = [f"pred_{g}" for g in labels]
    suffix = f"{condition}_seed{seed}{tag}"
    pd.DataFrame(cm, index=idx, columns=col).to_csv(f"{METRICS_DIR}/confusion_matrix_raw_{suffix}.csv")
    pd.DataFrame(cmn, index=idx, columns=col).to_csv(f"{METRICS_DIR}/confusion_matrix_normalized_{suffix}.csv")
    return cm, cmn

PRED_STORE = {}   # (condition, seed) -> dict(y_true, y_pred, patient_ids)
rows, collapse_report = [], []

def evaluate_condition(model, condition, seed, view_mode="dual", ckpt=None, quantization="FP32",
                       loader=None, on_cpu_model=False, with_shift=True, with_latency=True):
    loader = loader or TEST_LOADER
    if on_cpu_model:
        y_true, y_pred, p_cum, pids = [], [], [], []
        with torch.no_grad():
            for b in loader:
                mm, dd = b["macula"].cpu(), b["disc"].cpu()
                if view_mode == "dual":
                    p = model(mm, dd)["p_dual"]
                else:
                    which = "macula" if "macula" in view_mode else "disc"
                    p = model.forward_single(mm if which == "macula" else dd, which=which)["p"]
                y_pred.extend((p > 0.5).sum(1).tolist()); y_true.extend(b["label"].tolist())
                p_cum.append(p); pids.extend(b["patient_id"].tolist())
        y_true, y_pred, p_cum, pids = np.array(y_true), np.array(y_pred), torch.cat(p_cum, 0), np.array(pids)
    else:
        y_true, y_pred, p_cum, pids = get_predictions(model, loader, DEVICE, view_mode, return_patient_ids=True)

    m = compute_all_metrics(y_true, y_pred, p_cum)
    row = {"condition": condition, "seed": seed, "quantization": quantization,
           "view_mode": view_mode, **m}

    if view_mode == "dual" and not on_cpu_model:
        row.update(compute_dual_view_gain(model, loader, DEVICE))
        if with_shift:
            row.update(compute_shift_fidelity(TEACHER, model, loader, DEVICE))
    row["ParamCount"] = param_count(model)
    if ckpt and os.path.exists(ckpt): row["CheckpointSize_MB"] = file_size_mb(ckpt)
    if with_latency:
        row.update(benchmark_latency(model, _sample["macula"], _sample["disc"], view_mode,
                                     on_cpu=True, label=condition))

    warns = check_prediction_collapse(y_true, y_pred)
    if warns:
        collapse_report.append({"condition": condition, "seed": seed, "quantization": quantization,
                                "warnings": "; ".join(warns)})
        print(f"  !! {condition}|s{seed}|{quantization}: " + "; ".join(warns))
    save_predictions(condition, seed, y_true, y_pred, p_cum, pids, quantization)
    save_confusion(condition, seed, y_true, y_pred, tag=f"_{quantization}")
    PRED_STORE[(condition, seed, quantization)] = {"y_true": y_true, "y_pred": y_pred, "patient_ids": pids}
    rows.append(row)
    return row

# ---- Teacher ----
evaluate_condition(TEACHER, "teacher", "-", "dual", TEACHER_CKPT, with_shift=False)

# ---- Single-view baselines (also the EXTERNAL gain reference) ----
for cond, vm in [("macula_only", "macula_only"), ("disc_only", "disc_only")]:
    for s in SEEDS_BASELINE:
        ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
        if not os.path.exists(ck): continue
        mdl = load_student(ck); evaluate_condition(mdl, cond, s, vm, ck, with_shift=False); del mdl

# ---- Core dual-view conditions ----
for cond in CORE_CONDITIONS:
    for s in SEEDS_CORE:
        ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
        if not os.path.exists(ck): continue
        mdl = load_student(ck); evaluate_condition(mdl, cond, s, "dual", ck); del mdl

# ---- Ablations ----
for cond, seeds in [("abl_csd_raw_smoothl1", SEEDS_BASELINE), ("abl_csd_kl_softmax", SEEDS_BASELINE),
                    ("abl_csd_counterfactual", [PRIMARY_SEED])]:
    for s in seeds:
        ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
        if not os.path.exists(ck): continue
        mdl = load_student(ck); evaluate_condition(mdl, cond, s, "dual", ck); del mdl

RAW = pd.DataFrame(rows)
RAW.to_csv(f"{METRICS_DIR}/all_conditions_raw.csv", index=False)
print(f"\nEvaluated {len(RAW)} model-runs on the internal test set.")

In [ ]:
# ---- External dual-view gain, now that the independent single-view references exist ----
_indep_mac = RAW[(RAW.condition == "macula_only")]["QWK"].mean()
_indep_dsc = RAW[(RAW.condition == "disc_only")]["QWK"].mean()
RAW["DualViewGain_G_external"] = RAW.apply(
    lambda r: compute_external_gain(r["QWK"], _indep_mac, _indep_dsc)
    if r["view_mode"] == "dual" else np.nan, axis=1)
print(f"External gain reference: independent macula QWK={_indep_mac:.4f}, disc QWK={_indep_dsc:.4f}")

# ---- Quantized variants of M* (RQ2) ----
BEST_FP32_MODEL = load_student(BEST_FP32_CKPT)
fp32_row = evaluate_condition(BEST_FP32_MODEL, "best_fp32", BEST_SEED, "dual", BEST_FP32_CKPT,
                              quantization="FP32")
if PTQ_OK:
    evaluate_condition(PTQ_MODEL, "ptq_int8", BEST_SEED, "dual", None, "PTQ_INT8", on_cpu_model=True)
if QAT_OK:
    evaluate_condition(QAT_MODEL, "qat_int8", BEST_SEED, "dual", None, "QAT_INT8", on_cpu_model=True)

RAW = pd.DataFrame(rows)
RAW["DualViewGain_G_external"] = RAW.apply(
    lambda r: compute_external_gain(r["QWK"], _indep_mac, _indep_dsc)
    if r["view_mode"] == "dual" else np.nan, axis=1)
RAW.to_csv(f"{METRICS_DIR}/all_conditions_raw.csv", index=False)

if collapse_report:
    pd.DataFrame(collapse_report).to_csv(f"{METRICS_DIR}/prediction_collapse_warnings.csv", index=False)
    print(f"\n{len(collapse_report)} collapse warning(s) written -- see prediction_collapse_warnings.csv")
else:
    print("\nNo prediction-collapse warnings: every model predicts >2 distinct grades with recall above floor.")

_all_grades_ok = not any("NEVER predicted" in c["warnings"] for c in collapse_report)
record_gate("Gate3_StudentViability", _all_grades_ok,
            f"{len(collapse_report)} condition(s) flagged; intermediate grades "
            f"{'reachable' if _all_grades_ok else 'COLLAPSED for some conditions'}")

In [ ]:
# ---- Aggregate across seeds (mean/SD + median/IQR) ----
NUMERIC = [c for c in RAW.columns if RAW[c].dtype.kind in "fc" and c not in ("seed",)]
agg_mean = RAW.groupby("condition")[NUMERIC].agg(["mean", "std", "median",
                                                   lambda x: x.quantile(.75) - x.quantile(.25)])
agg_mean.columns = ["_".join([a, b if b != "<lambda_0>" else "iqr"]) for a, b in agg_mean.columns]
agg_mean.to_csv(f"{METRICS_DIR}/all_conditions_aggregated.csv")

summary_cols = ["QWK", "Accuracy", "MacroPrecision", "MacroRecall", "MacroF1", "MAE", "SevereErrorRate"]
tbl = RAW.groupby("condition")[summary_cols].agg(["mean", "std"]).round(4)
print(tbl.to_string())

## 34 — Statistical analysis

**Paired patient-clustered bootstrap**, B=10,000. Resamples cluster IDs (DRTiD's per-eye `ID`, the
finest key available) with replacement rather than individual images, because predictions from the
same record are not independent. Paired because all compared models predict the *same* samples,
which gives more power than treating them as independent groups.

Comparisons are **pre-registered** (declared in the config cell before any result was seen) and
Holm-corrected. Effect sizes with CIs are reported — not bare p-values.

In [ ]:
def _metric_fns():
    return {"QWK": lambda t, p: fast_qwk(t, p),
            "Accuracy": lambda t, p: float((t == p).mean()),
            "MacroF1": lambda t, p: float(f1_score(t, p, average="macro",
                                                    labels=list(range(NUM_CLASSES)), zero_division=0)),
            "MAE": lambda t, p: float(np.mean(np.abs(t - p)))}

def paired_cluster_bootstrap(a_key, b_key, B=BOOTSTRAP_B, alpha=BOOTSTRAP_ALPHA, rng_seed=0):
    """Returns per-metric mean difference (A-B) with a percentile CI."""
    A, Bp = PRED_STORE.get(a_key), PRED_STORE.get(b_key)
    if A is None or Bp is None: return None
    assert np.array_equal(A["y_true"], Bp["y_true"]), "paired bootstrap requires identical sample order"
    y = A["y_true"]; pa, pb = A["y_pred"], Bp["y_pred"]; clusters = A["patient_ids"]
    uniq = np.unique(clusters)
    idx_by_cluster = {c: np.where(clusters == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed)
    fns = _metric_fns()
    diffs = {k: np.empty(B) for k in fns}
    for b in range(B):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by_cluster[c] for c in pick])
        yt, ya, yb = y[idx], pa[idx], pb[idx]
        for k, fn in fns.items():
            diffs[k][b] = fn(yt, ya) - fn(yt, yb)
    out = {}
    for k, d in diffs.items():
        lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        # Two-sided bootstrap p: proportion of replicates on the other side of zero.
        p = 2 * min((d <= 0).mean(), (d >= 0).mean()); p = min(max(p, 1.0 / B), 1.0)
        out[k] = {"mean_diff": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi),
                  "excludes_zero": bool(lo > 0 or hi < 0), "p_boot": float(p)}
    return out

def holm_correction(pvals, names, alpha=0.05):
    """Holm-Bonferroni step-down. Adjusted p-values are enforced monotone non-decreasing along the
    sorted order, which is what makes the procedure valid."""
    pvals = list(pvals)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = [None] * m
    running = 0.0
    for rank, i in enumerate(order):
        val = min(1.0, (m - rank) * pvals[i])
        running = max(running, val)
        adj[i] = running
    return {names[i]: {"p_raw": float(pvals[i]), "p_holm": float(adj[i]),
                       "significant_holm": bool(adj[i] < alpha)} for i in range(m)}

stat_rows, p_for_holm, names_for_holm = [], [], []
for rq, pairs in PREREGISTERED_COMPARISONS.items():
    for a, b in pairs:
        a_key = (a, BEST_SEED if a in ("best_fp32", "ptq_int8", "qat_int8") else BEST_CSD_SEED,
                 {"ptq_int8": "PTQ_INT8", "qat_int8": "QAT_INT8"}.get(a, "FP32"))
        b_key = (b, BEST_SEED if b in ("best_fp32", "ptq_int8", "qat_int8") else SEEDS_CORE[0],
                 {"ptq_int8": "PTQ_INT8", "qat_int8": "QAT_INT8"}.get(b, "FP32"))
        res = paired_cluster_bootstrap(a_key, b_key)
        if res is None:
            print(f"  skip {rq}: {a} vs {b} (missing predictions)"); continue
        for metric, r in res.items():
            stat_rows.append({"RQ": rq, "comparison": f"{a}_vs_{b}", "metric": metric, **r})
            if metric == "QWK":
                p_for_holm.append(r["p_boot"]); names_for_holm.append(f"{rq}:{a}_vs_{b}")
        print(f"  {rq} {a} vs {b}: dQWK={res['QWK']['mean_diff']:+.4f} "
              f"[{res['QWK']['ci_low']:+.4f},{res['QWK']['ci_high']:+.4f}] "
              f"dMacroF1={res['MacroF1']['mean_diff']:+.4f}")

STATS = pd.DataFrame(stat_rows)
if len(p_for_holm):
    holm = holm_correction(p_for_holm, names_for_holm)
    STATS["holm_key"] = STATS.apply(lambda r: f"{r['RQ']}:{r['comparison']}", axis=1)
    STATS["p_holm"] = STATS.apply(lambda r: holm.get(r["holm_key"], {}).get("p_holm", np.nan)
                                  if r["metric"] == "QWK" else np.nan, axis=1)
    STATS["significant_holm"] = STATS.apply(lambda r: holm.get(r["holm_key"], {}).get("significant_holm", None)
                                            if r["metric"] == "QWK" else None, axis=1)
STATS.to_csv(f"{TABLES_DIR}/table_05_statistical_tests.csv", index=False)
print(f"\nSaved {len(STATS)} statistical comparisons (B={BOOTSTRAP_B}, Holm-corrected on QWK).")

## 35 — DeepDRiD external confirmatory validation (frozen)

Models are **frozen** before this section: no fine-tuning, no threshold tuning, no model selection,
no method change based on what happens here. A drop versus DRTiD is a domain-shift finding to
report, not something to engineer away.

**Documented ambiguity:** DeepDRiD's public CSVs contain no column stating which of `_1`/`_2` is
macula- vs disc-centred (`Field definition` is an image-quality score, not a field-type label).
Rather than bury an assumption, **both orderings are evaluated and both are reported** — turning
the unknown into a small robustness check.

In [ ]:
class DeepDRiDDualViewDataset(Dataset):
    """One record per EYE: (<pid>_<l|r>1, <pid>_<l|r>2) with that eye's DR level."""
    def __init__(self, root, subsets=("regular-fundus-training", "regular-fundus-validation"),
                 transform=None, field_order="_1=macula"):
        self.transform = transform or eval_transform
        self.field_order = field_order
        recs = []
        for sub in subsets:
            csv = f"{root}/{sub}/{sub.replace('regular-fundus-', 'regular-fundus-')}.csv"
            if not os.path.exists(csv): continue
            df = pd.read_csv(csv)
            img_root = f"{root}/{sub}/Images"
            for pid, grp in df.groupby("patient_id"):
                for eye, col in (("l", "left_eye_DR_Level"), ("r", "right_eye_DR_Level")):
                    sub_g = grp[grp["image_id"].astype(str).str.contains(f"_{eye}")]
                    if len(sub_g) < 2: continue
                    lvl = sub_g[col].dropna()
                    if lvl.empty: continue
                    grade = int(lvl.iloc[0])
                    if not (0 <= grade < NUM_CLASSES): continue
                    f1 = f"{img_root}/{pid}/{pid}_{eye}1.jpg"
                    f2 = f"{img_root}/{pid}/{pid}_{eye}2.jpg"
                    if os.path.exists(f1) and os.path.exists(f2):
                        recs.append({"patient_id": int(pid), "eye": eye, "f1": f1, "f2": f2, "grade": grade})
        self.df = pd.DataFrame(recs)

    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        a, b = (r["f1"], r["f2"]) if self.field_order == "_1=macula" else (r["f2"], r["f1"])
        return {"macula": self.transform(image=_rgb(a))["image"],
                "disc":   self.transform(image=_rgb(b))["image"],
                "label":  torch.tensor(int(r["grade"]), dtype=torch.long),
                "patient_id": int(r["patient_id"])}

EXTERNAL_ROWS = []
if DEEPDRID_ROOT is None:
    print("DeepDRiD not found on Drive -- external validation SKIPPED (reported, not silently omitted).")
    record_gate("Gate9_ExternalValidation", False, "DeepDRiD dataset not present in this runtime")
else:
    ext_models = [("teacher", "-", TEACHER, "FP32", False),
                  ("best_fp32", BEST_SEED, BEST_FP32_MODEL, "FP32", False)]
    if BEST_CONDITION != "dual_csd":
        ext_models.append(("best_csd_fp32", BEST_CSD_SEED, load_student(BEST_CSD_CKPT), "FP32", False))
    if PTQ_OK: ext_models.append(("ptq_int8", BEST_SEED, PTQ_MODEL, "PTQ_INT8", True))
    if QAT_OK: ext_models.append(("qat_int8", BEST_SEED, QAT_MODEL, "QAT_INT8", True))

    for order in DEEPDRID_FIELD_ORDERS:
        ds = DeepDRiDDualViewDataset(DEEPDRID_ROOT, transform=eval_transform, field_order=order)
        if len(ds) == 0:
            print(f"  DeepDRiD produced 0 usable eye-records for order {order}"); continue
        ld = make_loader(ds, 16, False, workers=2)
        print(f"\nDeepDRiD [{order}]: {len(ds)} eyes, {ds.df.patient_id.nunique()} patients, "
              f"grades {sorted(ds.df.grade.unique().tolist())}")
        for name, seed, mdl, quant, on_cpu in ext_models:
            if on_cpu:
                yt, yp, pc, pid = [], [], [], []
                with torch.no_grad():
                    for b in ld:
                        p = mdl(b["macula"].cpu(), b["disc"].cpu())["p_dual"]
                        yp.extend((p > 0.5).sum(1).tolist()); yt.extend(b["label"].tolist())
                        pc.append(p); pid.extend(b["patient_id"].tolist())
                yt, yp, pc, pid = np.array(yt), np.array(yp), torch.cat(pc, 0), np.array(pid)
            else:
                yt, yp, pc, pid = get_predictions(mdl, ld, DEVICE, "dual", return_patient_ids=True)
            m = compute_all_metrics(yt, yp, pc)
            EXTERNAL_ROWS.append({"condition": name, "seed": seed, "quantization": quant,
                                  "field_order": order, "n_eyes": len(ds), **m})
            save_predictions(name, seed, yt, yp, pc, pid, quant, dataset="DeepDRiD",
                             split=f"external_{order.replace('=', '')}")
            save_confusion(name, seed, yt, yp, tag=f"_DeepDRiD_{order.replace('=', '')}")
            w = check_prediction_collapse(yt, yp)
            print(f"  {name:14s} QWK={m['QWK']:.4f} Acc={m['Accuracy']:.4f} MacroF1={m['MacroF1']:.4f}"
                  + (f"  !! {'; '.join(w)}" if w else ""))

    if EXTERNAL_ROWS:
        EXT = pd.DataFrame(EXTERNAL_ROWS)
        EXT.to_csv(f"{TABLES_DIR}/table_06_external_validation_deepdrid.csv", index=False)
        record_gate("Gate9_ExternalValidation", True,
                    f"{EXT.condition.nunique()} models x {EXT.field_order.nunique()} field orders on DeepDRiD")
    else:
        record_gate("Gate9_ExternalValidation", False, "no usable DeepDRiD records assembled")

EXT_DF = pd.DataFrame(EXTERNAL_ROWS) if EXTERNAL_ROWS else pd.DataFrame()

## 36–37 — Deployment export, parity checks & inference wrapper (Gate 8)

TorchScript is deprecated and is **not** used as the deployment path. Artifacts:
`checkpoint.pt` (native), `model.pt2` (`torch.export`), `model.onnx`, `metadata.json`.
Export failures are reported as failures — never silently downgraded to "gate passed".

In [ ]:
class DRVergeInference(nn.Module):
    """Export-friendly wrapper: two images in, cumulative threshold scores out."""
    def __init__(self, model): super().__init__(); self.model = model
    def forward(self, macula, disc): return self.model(macula, disc)["p_dual"]

def export_model(model, name, on_cpu_model=False, extra_meta=None):
    d = f"{MODELS_DIR}/{name}"; os.makedirs(d, exist_ok=True)
    m = (model if on_cpu_model else copy.deepcopy(model).to("cpu")).eval()
    ex_m, ex_d = _sample["macula"][:1].cpu(), _sample["disc"][:1].cpu()
    status = {"state_dict": False, "torch_export": False, "onnx": False,
              "onnx_parity_max_abs_diff": None, "export_error": None, "onnx_error": None}

    try:
        robust_torch_save(m.state_dict(), f"{d}/checkpoint.pt"); status["state_dict"] = True
    except Exception as e:
        status["export_error"] = f"state_dict: {e!r}"

    wrapper = DRVergeInference(m).eval()
    with torch.no_grad():
        ref = wrapper(ex_m, ex_d)

    try:
        ep = torch.export.export(wrapper, (ex_m, ex_d))
        torch.export.save(ep, f"{d}/model.pt2"); status["torch_export"] = True
    except Exception as e:
        status["export_error"] = f"torch.export: {e!r}"
        print(f"  [{name}] torch.export FAILED: {e!r}")

    try:
        torch.onnx.export(wrapper, (ex_m, ex_d), f"{d}/model.onnx",
                          input_names=["macula", "disc"], output_names=["p_cumulative"], dynamo=True)
        status["onnx"] = True
        try:
            import onnxruntime as ort
            sess = ort.InferenceSession(f"{d}/model.onnx", providers=["CPUExecutionProvider"])
            got = sess.run(None, {"macula": ex_m.numpy(), "disc": ex_d.numpy()})[0]
            status["onnx_parity_max_abs_diff"] = float(np.max(np.abs(got - ref.numpy())))
        except Exception as e:
            status["onnx_error"] = f"parity: {e!r}"
    except Exception as e:
        status["onnx_error"] = f"export: {e!r}"
        print(f"  [{name}] ONNX export FAILED: {e!r}")

    meta = {"model_name": name, "architecture": type(m).__name__, "dataset": "DRTiD",
            "grade_mapping": list(range(NUM_CLASSES)), "num_thresholds": NUM_THRESHOLDS,
            **PREPROCESSING_META, "torch_version": torch.__version__,
            "torchao_version": ENVIRONMENT.get("torchao"), "git_commit": ENVIRONMENT["git_commit"],
            "artifacts": {k: v for k, v in status.items()}, **(extra_meta or {})}
    save_json(meta, f"{d}/metadata.json")
    ok = status["state_dict"]
    print(f"  [{name}] state_dict={status['state_dict']} pt2={status['torch_export']} "
          f"onnx={status['onnx']} parity={status['onnx_parity_max_abs_diff']}")
    return d, status, meta

EXPORTS = {}
EXPORTS["teacher_fp32"] = export_model(TEACHER, "teacher_fp32",
    extra_meta={"role": "upper_bound_teacher", "training_seed": PRIMARY_SEED})
EXPORTS["best_student_fp32"] = export_model(BEST_FP32_MODEL, "best_student_fp32",
    extra_meta={"role": "deployment_candidate", "condition": BEST_CONDITION,
                "training_seed": BEST_SEED, "best_val_qwk": float(BEST_ROW["QWK"]), "quantization": "FP32"})
if BEST_CONDITION != "dual_csd":
    EXPORTS["best_csd_fp32"] = export_model(load_student(BEST_CSD_CKPT), "best_csd_fp32",
        extra_meta={"role": "best_csd_artifact", "condition": "dual_csd",
                    "training_seed": BEST_CSD_SEED, "quantization": "FP32"})
if PTQ_OK:
    EXPORTS["best_student_ptq_int8"] = export_model(PTQ_MODEL, "best_student_ptq_int8", on_cpu_model=True,
        extra_meta={"role": "deployment_int8", "quantization": "PTQ_INT8", **PTQ_INFO})
if QAT_OK:
    EXPORTS["best_student_qat_int8"] = export_model(QAT_MODEL, "best_student_qat_int8", on_cpu_model=True,
        extra_meta={"role": "deployment_int8", "quantization": "QAT_INT8", **QAT_INFO})

_export_ok = all(s["state_dict"] for _, s, _ in EXPORTS.values())
_pt2_ok = sum(1 for _, s, _ in EXPORTS.values() if s["torch_export"])
record_gate("Gate8_Export", _export_ok,
            f"{len(EXPORTS)} models; state_dict all={_export_ok}; torch.export ok for {_pt2_ok}/{len(EXPORTS)}")

In [ ]:
# ---- Deployment verification (spec 59): reload from disk and confirm it still behaves ----
def verify_deployment(model_dir, reference_model, on_cpu_model=False, n_runs=100):
    checks = {}
    ex_m, ex_d = _sample["macula"][:1].cpu(), _sample["disc"][:1].cpu()
    ref = (reference_model if on_cpu_model else copy.deepcopy(reference_model).to("cpu")).eval()
    with torch.no_grad():
        ref_out = DRVergeInference(ref).eval()(ex_m, ex_d)
    checks["reference_grade_in_range"] = bool(0 <= int((ref_out > 0.5).sum()) <= NUM_CLASSES - 1)
    try:
        with torch.no_grad():
            for _ in range(n_runs): DRVergeInference(ref).eval()(ex_m, ex_d)
        checks["stable_over_100_runs"] = True
    except Exception as e:
        checks["stable_over_100_runs"] = False; checks["run_error"] = repr(e)
    pt2 = f"{model_dir}/model.pt2"
    if os.path.exists(pt2):
        try:
            loaded = torch.export.load(pt2)
            with torch.no_grad(): got = loaded.module()(ex_m, ex_d)
            checks["pt2_reload_max_abs_diff"] = float(torch.max(torch.abs(got - ref_out)))
            checks["pt2_parity_ok"] = checks["pt2_reload_max_abs_diff"] < 1e-4
        except Exception as e:
            checks["pt2_parity_ok"] = False; checks["pt2_error"] = repr(e)
    return checks

DEPLOY_CHECKS = {}
DEPLOY_CHECKS["best_student_fp32"] = verify_deployment(EXPORTS["best_student_fp32"][0], BEST_FP32_MODEL)
if PTQ_OK: DEPLOY_CHECKS["best_student_ptq_int8"] = verify_deployment(EXPORTS["best_student_ptq_int8"][0], PTQ_MODEL, True)
if QAT_OK: DEPLOY_CHECKS["best_student_qat_int8"] = verify_deployment(EXPORTS["best_student_qat_int8"][0], QAT_MODEL, True)
save_json(DEPLOY_CHECKS, f"{RESULTS_DIR}/deployment_verification.json")
print(json.dumps(DEPLOY_CHECKS, indent=2, default=str))

In [ ]:
# ---- Final inference interface (spec 21) -- what a web prototype would call ----
GRADE_NAMES = {0: "No DR", 1: "Mild NPDR", 2: "Moderate NPDR", 3: "Severe NPDR", 4: "Proliferative DR"}

def predict_dr(macula_image, optic_disc_image, model=None, model_version=None, device="cpu"):
    """macula_image / optic_disc_image: HxWx3 uint8 RGB arrays or file paths."""
    model = model if model is not None else BEST_FP32_MODEL
    version = model_version or f"{BEST_CONDITION}_seed{BEST_SEED}_FP32"
    m = copy.deepcopy(model).to(device).eval()
    def prep(x):
        arr = _rgb(x) if isinstance(x, str) else np.asarray(x)
        return eval_transform(image=arr)["image"].unsqueeze(0).to(device)
    a, b = prep(macula_image), prep(optic_disc_image)
    t0 = time.perf_counter()
    with torch.no_grad():
        p = m(a, b)["p_dual"][0]
    dt = (time.perf_counter() - t0) * 1000
    cum = p.cpu().numpy()
    grade = int((cum > 0.5).sum())
    # cumulative P(y>k) -> per-class probabilities
    ext = np.concatenate([[1.0], cum, [0.0]])
    probs = np.clip(ext[:-1] - ext[1:], 0, None); probs = probs / max(probs.sum(), 1e-9)
    return {"predicted_grade": grade, "predicted_label": GRADE_NAMES[grade],
            "cumulative_threshold_scores": cum.tolist(), "grade_probabilities": probs.tolist(),
            "confidence": float(probs[grade]), "model_version": version,
            "inference_time_ms": dt, "preprocessing": PREPROCESSING_META}

_demo = pd.read_csv(DRTID_TEST_CSV).iloc[0]
_out = predict_dr(_demo["macula_path"], _demo["disc_path"])
print("predict_dr() demo ->", json.dumps({k: v for k, v in _out.items() if k != "preprocessing"},
                                          indent=2, default=str))
print("true grade:", int(_demo["grade"]))

In [ ]:
# ---- Model registry (spec 58) ----
reg = []
_ext_lookup = {}
if len(EXT_DF):
    for _, r in EXT_DF[EXT_DF.field_order == DEEPDRID_FIELD_ORDERS[0]].iterrows():
        _ext_lookup[r["condition"]] = r["QWK"]

for name, (d, status, meta) in EXPORTS.items():
    cond = meta.get("condition", meta.get("role", name))
    quant = meta.get("quantization", "FP32")
    match = RAW[(RAW.condition == {"teacher_fp32": "teacher", "best_student_fp32": "best_fp32",
                                   "best_student_ptq_int8": "ptq_int8", "best_student_qat_int8": "qat_int8",
                                   "best_csd_fp32": "dual_csd"}.get(name, name))]
    if name == "best_csd_fp32": match = match[match.seed == BEST_CSD_SEED]
    row = match.iloc[0] if len(match) else None
    reg.append({
        "model_id": name, "condition": cond, "seed": meta.get("training_seed", "-"),
        "checkpoint_path": f"{d}/checkpoint.pt",
        "pt2_path": f"{d}/model.pt2" if status["torch_export"] else "",
        "onnx_path": f"{d}/model.onnx" if status["onnx"] else "",
        "val_qwk": meta.get("best_val_qwk", np.nan),
        "test_qwk": float(row["QWK"]) if row is not None else np.nan,
        "test_macro_f1": float(row["MacroF1"]) if row is not None else np.nan,
        "test_accuracy": float(row["Accuracy"]) if row is not None else np.nan,
        "external_qwk": _ext_lookup.get({"teacher_fp32": "teacher", "best_student_fp32": "best_fp32",
                                          "best_student_ptq_int8": "ptq_int8",
                                          "best_student_qat_int8": "qat_int8"}.get(name, name), np.nan),
        "params": int(row["ParamCount"]) if row is not None and not pd.isna(row.get("ParamCount")) else np.nan,
        "size_mb": file_size_mb(f"{d}/checkpoint.pt"),
        "latency_median_ms": float(row["Latency_median_ms"]) if row is not None and "Latency_median_ms" in row else np.nan,
        "quantization": quant,
        "deployable": bool(status["state_dict"] and (name not in DEPLOY_CHECKS or
                                                     DEPLOY_CHECKS[name].get("stable_over_100_runs", False))),
    })
REGISTRY = pd.DataFrame(reg)
REGISTRY.to_csv(f"{ART}/model_registry.csv", index=False)
print(REGISTRY.to_string(index=False))

## 38 — Figures & companion CSVs

Every figure is written as **PNG (400 dpi) + PDF + SVG**, and every figure ships a
`*_data.csv` with exactly the numbers plotted. No value exists only inside an image.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 400, "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True,
                     "figure.autolayout": False})

def save_figure(fig, stem, data_df, caption=""):
    for ext in ("png", "pdf", "svg"):
        fig.savefig(f"{FIGURES_DIR}/{stem}.{ext}", bbox_inches="tight")
    data_df.to_csv(f"{FIGURES_DIR}/{stem}_data.csv", index=False)
    if caption:
        with open(f"{FIGURES_DIR}/{stem}_caption.txt", "w") as f: f.write(caption)
    plt.close(fig)
    print(f"  saved {stem} (.png/.pdf/.svg + _data.csv)")

def order_present(order, df=None):
    df = RAW if df is None else df
    return [c for c in order if c in set(df["condition"])]

DISPLAY_ORDER = ["teacher", "macula_only", "disc_only", "dual_no_distill", "dual_logitkd",
                 "dual_featkd", "dual_csd", "abl_csd_raw_smoothl1", "abl_csd_kl_softmax",
                 "abl_csd_counterfactual", "best_fp32", "ptq_int8", "qat_int8"]

def agg_stat(df, conds, col):
    means, sds, ns = [], [], []
    for c in conds:
        v = df[df.condition == c][col].dropna()
        means.append(v.mean() if len(v) else np.nan)
        sds.append(v.std() if len(v) > 1 else 0.0)
        ns.append(len(v))
    return np.array(means), np.array(sds), np.array(ns)

In [ ]:
# ---- Figure 1: architecture / workflow schematic (hero figure) ----
fig, ax = plt.subplots(figsize=(11, 6)); ax.axis("off"); ax.grid(False)
def box(x, y, w, h, txt, fc):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=fc, edgecolor="#333", lw=1.4, zorder=2))
    ax.text(x + w/2, y + h/2, txt, ha="center", va="center", fontsize=10, zorder=3)
def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.6, color="#333"))

box(0.02, 0.72, 0.15, 0.10, "Macula view", "#DCE9F7"); box(0.02, 0.58, 0.15, 0.10, "Optic-disc view", "#DCE9F7")
box(0.24, 0.62, 0.20, 0.20, "Teacher\nResNet-50 dual-view\n+ CORAL heads", "#F7DCDC")
box(0.50, 0.72, 0.24, 0.10, "p_dual, p_macula, p_disc", "#FFF3CD")
box(0.50, 0.58, 0.24, 0.10, r"$\Delta^T=p_{dual}-p_{agg}$", "#FFF3CD")
box(0.24, 0.30, 0.20, 0.18, "Lightweight student\ndepthwise-separable\n+ InteractionFusion", "#DCF7E3")
box(0.50, 0.34, 0.24, 0.10, "L = task + aux\n+ α·KD + β·CSD", "#FFF3CD")
box(0.80, 0.46, 0.17, 0.09, "Best FP32 (M*)", "#DCF7E3")
box(0.80, 0.32, 0.17, 0.09, "PTQ INT8", "#E8DCF7"); box(0.80, 0.18, 0.17, 0.09, "QAT INT8", "#E8DCF7")
box(0.02, 0.30, 0.15, 0.10, "Macula view", "#DCE9F7"); box(0.02, 0.16, 0.15, 0.10, "Optic-disc view", "#DCE9F7")
arrow(0.17, 0.77, 0.24, 0.74); arrow(0.17, 0.63, 0.24, 0.68)
arrow(0.44, 0.74, 0.50, 0.77); arrow(0.44, 0.68, 0.50, 0.63)
arrow(0.17, 0.35, 0.24, 0.40); arrow(0.17, 0.21, 0.24, 0.36)
arrow(0.44, 0.39, 0.50, 0.39); arrow(0.62, 0.58, 0.62, 0.44)
arrow(0.74, 0.39, 0.80, 0.50); arrow(0.885, 0.46, 0.885, 0.41); arrow(0.885, 0.32, 0.885, 0.27)
ax.text(0.63, 0.535, "CSD", fontsize=9, ha="center", color="#B03A2E")
ax.set_xlim(0, 1); ax.set_ylim(0.1, 0.9)
ax.set_title("Figure 1 — DR-VERGE: complementarity-shift distillation and INT8 deployment", fontsize=12)
save_figure(fig, "fig_01_architecture",
            pd.DataFrame([{"component": "teacher", "detail": "ResNet-50 dual-view + CORAL"},
                          {"component": "student", "detail": f"depthwise-separable {STUDENT_CHANNELS}"},
                          {"component": "distillation", "detail": "task + aux + logit-KD + CSD"},
                          {"component": "deployment", "detail": "FP32 / PTQ INT8 / QAT INT8"}]),
            "DR-VERGE architecture. Teacher produces dual and single-view cumulative probabilities; "
            "their difference is the complementarity shift distilled into the lightweight student.")

In [ ]:
# ---- Figure 3: predictive performance comparison (QWK / Macro-F1 / Accuracy) ----
conds = order_present(["teacher", "macula_only", "disc_only", "dual_no_distill",
                       "dual_logitkd", "dual_featkd", "dual_csd"])
metrics3 = ["QWK", "MacroF1", "Accuracy"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
rows3 = []
for ax, met in zip(axes, metrics3):
    mu, sd, ns = agg_stat(RAW, conds, met)
    ax.bar(range(len(conds)), mu, yerr=sd, capsize=4, color="#4C72B0", edgecolor="#25405e")
    for i, c in enumerate(conds):
        pts = RAW[RAW.condition == c][met].dropna().values
        if len(pts) > 1: ax.scatter([i] * len(pts), pts, s=16, color="#C44E52", zorder=3, alpha=.85)
    ax.set_xticks(range(len(conds))); ax.set_xticklabels(conds, rotation=35, ha="right", fontsize=9)
    ax.set_title(f"{met} (higher is better)"); ax.set_ylabel(met)
    for c, m_, s_, n_ in zip(conds, mu, sd, ns):
        rows3.append({"metric": met, "condition": c, "mean": m_, "sd": s_, "n_seeds": n_})
fig.suptitle("Figure 3 — Predictive performance on the internal DRTiD test set (mean ± SD over seeds; dots = individual seeds)", y=1.02)
save_figure(fig, "fig_03_performance_comparison", pd.DataFrame(rows3),
            "Predictive performance. Error bars are SD across seeds; red dots are individual seed values.")

In [ ]:
# ---- Figure 4: efficiency Pareto frontier ----
eff_conds = order_present(["teacher", "best_fp32", "ptq_int8", "qat_int8"])
rows4 = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (xcol, xlabel) in zip(axes, [("Latency_median_ms", "CPU latency, median (ms) — lower is better"),
                                     ("CheckpointSize_MB", "Serialized size (MB) — lower is better")]):
    for c in eff_conds:
        sub = RAW[RAW.condition == c]
        if not len(sub) or xcol not in sub: continue
        x, y = sub[xcol].mean(), sub["QWK"].mean()
        if pd.isna(x): continue
        mk = "D" if c == "teacher" else ("*" if "int8" in c else "o")
        ax.scatter(x, y, s=320 if mk != "o" else 150, marker=mk, label=c, edgecolor="#222", zorder=3)
        ax.annotate(c, (x, y), textcoords="offset points", xytext=(8, 6), fontsize=9)
        rows4.append({"axis": xcol, "condition": c, "x": x, "QWK": y})
    ax.set_xlabel(xlabel); ax.set_ylabel("QWK (higher is better)"); ax.set_xscale("log")
    ax.set_title("Top-left is better")
fig.suptitle("Figure 4 — Efficiency–accuracy Pareto frontier", y=1.02)
save_figure(fig, "fig_04_efficiency_pareto", pd.DataFrame(rows4),
            "Efficiency frontier: QWK against CPU latency and serialized size (log x-axis).")

In [ ]:
# ---- Figure 5: quantization retention (FP32 vs PTQ vs QAT) ----
qconds = order_present(["best_fp32", "ptq_int8", "qat_int8"])
qmetrics = ["QWK", "Accuracy", "MacroF1", "MacroRecall"]
rows5 = []
if len(qconds) >= 2:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    w = 0.8 / len(qconds)
    for i, c in enumerate(qconds):
        vals = [RAW[RAW.condition == c][m].mean() for m in qmetrics]
        ax.bar(np.arange(len(qmetrics)) + i * w, vals, width=w, label=c, edgecolor="#25405e")
        for m, v in zip(qmetrics, vals): rows5.append({"condition": c, "metric": m, "value": v})
    ax.set_xticks(np.arange(len(qmetrics)) + w * (len(qconds) - 1) / 2); ax.set_xticklabels(qmetrics)
    ax.set_ylabel("score (higher is better)"); ax.legend()
    ax.set_title("Figure 5 — Quantization: FP32 vs PTQ INT8 vs QAT INT8")
    save_figure(fig, "fig_05_quantization_retention", pd.DataFrame(rows5),
                "Diagnostic performance retained after INT8 quantization.")
else:
    print("  fig_05 skipped: fewer than two quantization variants available")

In [ ]:
# ---- Figure 6: per-grade sensitivity (the rev2 failure mode made permanently visible) ----
pg_conds = order_present(["teacher", "best_fp32", "dual_csd", "ptq_int8", "qat_int8"])
rows6 = []
fig, ax = plt.subplots(figsize=(11, 5.5))
w = 0.8 / max(len(pg_conds), 1)
for i, c in enumerate(pg_conds):
    vals = [RAW[RAW.condition == c][f"Sensitivity_Grade{g}"].mean() for g in range(NUM_CLASSES)]
    ax.bar(np.arange(NUM_CLASSES) + i * w, vals, width=w, label=c, edgecolor="#25405e")
    for g, v in enumerate(vals): rows6.append({"condition": c, "grade": g, "sensitivity": v})
ax.axhline(0.05, color="#C44E52", ls="--", lw=1.2, label="collapse floor (0.05)")
ax.set_xticks(np.arange(NUM_CLASSES) + w * (len(pg_conds) - 1) / 2)
ax.set_xticklabels([f"Grade {g}" for g in range(NUM_CLASSES)])
ax.set_ylabel("Sensitivity / recall (higher is better)"); ax.set_ylim(0, 1); ax.legend(fontsize=9)
ax.set_title("Figure 6 — Per-grade sensitivity: are intermediate grades actually predicted?")
save_figure(fig, "fig_06_per_grade_sensitivity", pd.DataFrame(rows6),
            "Per-grade recall. Grades 1-3 near zero indicates the ordinal collapse seen in rev2.")

In [ ]:
# ---- Figure 7: normalized confusion matrices ----
cm_conds = order_present(["teacher", "best_fp32", "ptq_int8", "qat_int8"])
rows7 = []
if cm_conds:
    fig, axes = plt.subplots(1, len(cm_conds), figsize=(4.6 * len(cm_conds), 4.4))
    if len(cm_conds) == 1: axes = [axes]
    for ax, c in zip(axes, cm_conds):
        key = next((k for k in PRED_STORE if k[0] == c), None)
        if key is None: ax.axis("off"); continue
        d = PRED_STORE[key]
        cm = confusion_matrix(d["y_true"], d["y_pred"], labels=list(range(NUM_CLASSES)))
        cmn = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1)
        im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
        ax.set_title(c, fontsize=11); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if cmn[i, j] > .5 else "black")
                rows7.append({"condition": c, "true": i, "pred": j, "count": int(cm[i, j]),
                              "normalized": float(cmn[i, j])})
    fig.suptitle("Figure 7 — Row-normalized confusion matrices (internal test set)", y=1.03)
    save_figure(fig, "fig_07_confusion_matrices", pd.DataFrame(rows7),
                "Row-normalized confusion matrices; diagonal = per-grade recall.")

In [ ]:
# ---- Figure 8: CSD mechanism (the primary RQ1 mechanism figure) ----
mech_conds = order_present(["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd",
                            "abl_csd_counterfactual"])
mech = ["ShiftMAE", "CosAgree", "BenefitCorr"]
rows8 = []
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, met in zip(axes, mech):
    mu, sd, ns = agg_stat(RAW, mech_conds, met)
    better = "lower is better" if met == "ShiftMAE" else "higher is better"
    ax.bar(range(len(mech_conds)), mu, yerr=sd, capsize=4, color="#55A868", edgecolor="#2f5d3f")
    ax.set_xticks(range(len(mech_conds))); ax.set_xticklabels(mech_conds, rotation=35, ha="right", fontsize=9)
    ax.set_title(f"{met} ({better})"); ax.set_ylabel(met)
    for c, m_, s_, n_ in zip(mech_conds, mu, sd, ns):
        rows8.append({"metric": met, "condition": c, "mean": m_, "sd": s_, "n_seeds": n_})
fig.suptitle("Figure 8 — CSD mechanism: is the teacher's complementarity shift actually transferred?", y=1.02)
save_figure(fig, "fig_08_csd_mechanism", pd.DataFrame(rows8),
            "Mechanism fidelity. ShiftMAE lower = student shift closer to teacher; CosAgree higher = same "
            "shift direction; BenefitCorr higher = student gains from dual-view on the same samples as the teacher.")

In [ ]:
# ---- Figure 9: gradient contributions over training ----
rows9 = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cond in ["dual_csd", "dual_logitkd", "dual_no_distill", "dual_featkd"]:
    f = f"{LOGS_DIR}/gradient_contributions_{cond}_{PRIMARY_SEED}.csv"
    if not os.path.exists(f): continue
    h = pd.read_csv(f)
    if "gnorm_task" in h:
        axes[0].plot(h["epoch"], h["gnorm_task"], label=f"{cond}: task", lw=1.4)
    if "gnorm_csd" in h:
        axes[0].plot(h["epoch"], h["gnorm_csd"], label=f"{cond}: CSD", lw=1.8, ls="--")
    if "gnorm_ratio_csd_over_task" in h:
        axes[1].plot(h["epoch"], h["gnorm_ratio_csd_over_task"], label=cond, lw=1.8)
    for _, r in h.iterrows():
        rows9.append({"condition": cond, **{k: r[k] for k in h.columns}})
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("gradient L2 norm"); axes[0].set_yscale("log")
axes[0].set_title("Per-component gradient norm"); axes[0].legend(fontsize=8)
axes[1].axhline(1.0, color="#888", ls=":", lw=1)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("‖∇L_CSD‖ / ‖∇L_task‖")
axes[1].set_title("CSD gradient share (rev2 was ≈0 — CSD had no influence)"); axes[1].legend(fontsize=8)
fig.suptitle("Figure 9 — Optimization signal: does CSD actually contribute gradient?", y=1.02)
save_figure(fig, "fig_09_gradient_contribution", pd.DataFrame(rows9),
            "Gradient norms per loss component. The ratio panel shows whether CSD is a real training "
            "signal or numerical decoration.")

In [ ]:
# ---- Figure 10: reliability diagrams (calibration) ----
rel_conds = order_present(["best_fp32", "ptq_int8", "qat_int8"])
rows10 = []
if rel_conds:
    fig, ax = plt.subplots(figsize=(7, 6.5))
    ax.plot([0, 1], [0, 1], ls="--", color="#888", label="perfectly calibrated")
    for c in rel_conds:
        key = next((k for k in PRED_STORE if k[0] == c), None)
        pf = f"{PREDS_DIR}/DRTiD_test_{c}_seed{BEST_SEED}_{ {'ptq_int8':'PTQ_INT8','qat_int8':'QAT_INT8'}.get(c,'FP32') }.csv"
        if not os.path.exists(pf): continue
        dfp = pd.read_csv(pf)
        p = torch.tensor(dfp[[f"p_threshold_{k}" for k in range(NUM_THRESHOLDS)]].values, dtype=torch.float32)
        rc = reliability_curve(p, dfp["true_grade"].values)
        ax.plot(rc["mean_predicted"], rc["observed_frequency"], marker="o", lw=1.8, label=c)
        cal = compute_calibration(p, dfp["true_grade"].values)
        for _, r in rc.iterrows():
            rows10.append({"condition": c, **r.to_dict(), "ECE": cal["ECE"], "Brier": cal["Brier"]})
    ax.set_xlabel("Mean predicted P(y>k)"); ax.set_ylabel("Observed frequency")
    ax.set_title("Figure 10 — Reliability diagram (pooled cumulative thresholds)")
    ax.legend(fontsize=9)
    save_figure(fig, "fig_10_calibration_reliability", pd.DataFrame(rows10),
                "Reliability diagram. Deviation from the diagonal indicates miscalibration; "
                "companion CSV carries ECE and Brier per condition.")
print(f"\nAll figures written to {FIGURES_DIR}")

## 39 — Final research tables

In [ ]:
# table_diagnostic_performance
diag_cols = ["QWK", "Accuracy", "MacroPrecision", "MacroRecall", "MacroF1", "WeightedF1",
             "BalancedAccuracy", "MAE", "SevereErrorRate", "ECE", "Brier"]
t_diag = RAW.groupby("condition")[diag_cols].agg(["mean", "std"]).round(4)
t_diag.columns = ["_".join(c) for c in t_diag.columns]
t_diag = t_diag.reindex([c for c in DISPLAY_ORDER if c in t_diag.index])
t_diag.to_csv(f"{TABLES_DIR}/table_diagnostic_performance.csv")

# table_efficiency
eff_cols = [c for c in ["ParamCount", "CheckpointSize_MB", "Latency_mean_ms", "Latency_median_ms",
                        "Latency_sd_ms", "Latency_p95_ms", "Latency_p99_ms", "Throughput_img_per_s"]
            if c in RAW.columns]
t_eff = RAW.groupby("condition")[eff_cols].mean().round(4)
ref_lat = t_eff.loc["teacher", "Latency_median_ms"] if "teacher" in t_eff.index else np.nan
ref_size = t_eff.loc["teacher", "CheckpointSize_MB"] if "teacher" in t_eff.index else np.nan
t_eff["Speedup_vs_teacher"] = (ref_lat / t_eff["Latency_median_ms"]).round(2)
t_eff["CompressionRatio_vs_teacher"] = (ref_size / t_eff["CheckpointSize_MB"]).round(2)
t_eff["SizeReduction_vs_teacher_pct"] = ((1 - t_eff["CheckpointSize_MB"] / ref_size) * 100).round(2)
t_eff = t_eff.reindex([c for c in DISPLAY_ORDER if c in t_eff.index])
t_eff.to_csv(f"{TABLES_DIR}/table_efficiency.csv")

# table_quantization (FP32 vs PTQ vs QAT) + retention
qrows = []
fp32_m = RAW[RAW.condition == "best_fp32"][diag_cols].mean().to_dict() if "best_fp32" in set(RAW.condition) else {}
for c in ["best_fp32", "ptq_int8", "qat_int8"]:
    if c not in set(RAW.condition): continue
    m = RAW[RAW.condition == c][diag_cols].mean().to_dict()
    e = RAW[RAW.condition == c][eff_cols].mean().to_dict()
    row = {"model": c, **{k: round(v, 4) for k, v in m.items()}, **{k: round(v, 4) for k, v in e.items()}}
    if fp32_m and c != "best_fp32":
        row.update({k: round(v, 3) for k, v in retention_metrics(m, fp32_m).items()})
        row.update(efficiency_derived(e.get("CheckpointSize_MB"), fp32_m and RAW[RAW.condition=="best_fp32"]["CheckpointSize_MB"].mean(),
                                      e.get("Latency_median_ms"), RAW[RAW.condition=="best_fp32"]["Latency_median_ms"].mean()))
    qrows.append(row)
t_quant = pd.DataFrame(qrows)
t_quant.to_csv(f"{TABLES_DIR}/table_quantization.csv", index=False)

# table_csd_mechanism
mech_cols = ["ShiftMAE", "CosAgree", "BenefitCorr", "DualViewGain_G_internal", "DualViewGain_G_external"]
t_mech = RAW[RAW.view_mode == "dual"].groupby("condition")[
    [c for c in mech_cols if c in RAW.columns]].agg(["mean", "std"]).round(4)
t_mech.columns = ["_".join(c) for c in t_mech.columns]
t_mech.to_csv(f"{TABLES_DIR}/table_csd_mechanism.csv")

print("Diagnostic performance:\n", t_diag[[c for c in t_diag.columns if c.endswith("_mean")]].to_string())
print("\nEfficiency:\n", t_eff.to_string())
print("\nQuantization:\n", t_quant.to_string(index=False))
print("\nCSD mechanism:\n", t_mech.to_string())

## 40–41 — Automatic headline generator & final gate report

In [ ]:
def _mean(cond, col):
    v = RAW[RAW.condition == cond][col].dropna()
    return float(v.mean()) if len(v) else float("nan")

headlines = []
t_q, s_q = _mean("teacher", "QWK"), _mean("best_fp32", "QWK")
t_p, s_p = _mean("teacher", "ParamCount"), _mean("best_fp32", "ParamCount")
t_l, s_l = _mean("teacher", "Latency_median_ms"), _mean("best_fp32", "Latency_median_ms")
t_s, s_s = _mean("teacher", "CheckpointSize_MB"), _mean("best_fp32", "CheckpointSize_MB")
if not math.isnan(t_p) and s_p:
    headlines.append(f"Teacher -> Student: {t_p/s_p:.0f}x fewer parameters, {t_s/s_s:.1f}x smaller artifact, "
                     f"{t_l/s_l:.1f}x faster CPU inference, {100*s_q/t_q:.1f}% of teacher QWK retained")
for q in ["ptq_int8", "qat_int8"]:
    if q not in set(RAW.condition): continue
    qq, ql, qs = _mean(q, "QWK"), _mean(q, "Latency_median_ms"), _mean(q, "CheckpointSize_MB")
    headlines.append(f"FP32 -> {q.upper()}: {100*qq/s_q:.1f}% QWK retained, "
                     f"{s_l/ql:.2f}x CPU speedup, {s_s/qs:.2f}x smaller"
                     if not math.isnan(qs) and qs else
                     f"FP32 -> {q.upper()}: {100*qq/s_q:.1f}% QWK retained, {s_l/ql:.2f}x CPU speedup")

print("=" * 78); print("AUTOMATIC HEADLINES (computed, NOT significance claims)"); print("=" * 78)
for h in headlines: print("  " + h)
pd.DataFrame({"headline": headlines}).to_csv(f"{TABLES_DIR}/table_headlines.csv", index=False)

In [ ]:
# ---- Gate 5: RQ1 verdict on predictive AND mechanistic axes ----
print("=" * 78); print("GATE 5 -- RQ1 VERDICT"); print("=" * 78)
csd_q = _mean("dual_csd", "QWK")
verdict = {}
for base in ["dual_no_distill", "dual_logitkd", "dual_featkd"]:
    if base not in set(RAW.condition): continue
    bq = _mean(base, "QWK")
    st = STATS[(STATS.comparison == f"dual_csd_vs_{base}") & (STATS.metric == "QWK")]
    ci = f"[{st.iloc[0]['ci_low']:+.4f}, {st.iloc[0]['ci_high']:+.4f}]" if len(st) else "n/a"
    cred = bool(st.iloc[0]["excludes_zero"]) if len(st) else None
    verdict[base] = {"csd_qwk": csd_q, "baseline_qwk": bq, "diff": csd_q - bq,
                     "ci_95": ci, "credible": cred}
    print(f"  CSD vs {base:18s}: {csd_q:.4f} vs {bq:.4f} (diff {csd_q-bq:+.4f}, 95% CI {ci}, credible={cred})")

print("\n  Mechanism (does CSD transfer the shift, independently of QWK?)")
for c in order_present(["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd"]):
    print(f"    {c:18s} ShiftMAE={_mean(c,'ShiftMAE'):.4f}  CosAgree={_mean(c,'CosAgree'):+.4f}  "
          f"BenefitCorr={_mean(c,'BenefitCorr'):+.4f}")
save_json(verdict, f"{RESULTS_DIR}/rq1_verdict.json")
record_gate("Gate5_RQ1_Comparison", True, "RQ1 comparisons computed on predictive and mechanistic axes")
print("\n  A negative or null RQ1 result is a valid, reportable finding -- do not re-tune the")
print("  method in response to this table. The protocol was locked before the run.")

In [ ]:
# ---- Final consolidated gate report ----
gate_df = pd.DataFrame([{"gate": k, "passed": v["passed"], "detail": v["detail"]} for k, v in GATES.items()])
gate_df.to_csv(f"{TABLES_DIR}/table_gate_report.csv", index=False)
print("=" * 78); print("FINAL GATE REPORT"); print("=" * 78)
print(gate_df.to_string(index=False))
n_pass = int(gate_df.passed.sum())
print(f"\n{n_pass}/{len(gate_df)} gates passed.")
failed = gate_df[~gate_df.passed]
if len(failed):
    print("\nFAILED / NOT-RUN gates (report these honestly rather than hiding them):")
    for _, r in failed.iterrows(): print(f"  - {r['gate']}: {r['detail']}")

RUN_SUMMARY = {
    "environment": ENVIRONMENT, "config": CONFIG_SNAPSHOT,
    "selection": {"best_condition": BEST_CONDITION, "best_seed": BEST_SEED,
                  "best_val_qwk": float(BEST_ROW["QWK"]), "best_csd_seed": BEST_CSD_SEED},
    "csd_selected": {"variant": BEST_CSD_VARIANT, "alpha": BEST_ALPHA, "beta": BEST_BETA},
    "gates": GATES, "headlines": headlines, "rq1_verdict": verdict,
    "n_evaluated_runs": int(len(RAW)),
    "external_validation": "completed" if len(EXT_DF) else "skipped/unavailable",
}
save_json(RUN_SUMMARY, f"{RESULTS_DIR}/run_summary.json")

print(f"""
{'='*78}
ARTIFACTS
{'='*78}
  checkpoints : {CKPT_DIR}
  models      : {MODELS_DIR}   (checkpoint.pt / model.pt2 / model.onnx / metadata.json)
  figures     : {FIGURES_DIR}  (png+pdf+svg + *_data.csv per figure)
  tables      : {TABLES_DIR}
  metrics     : {METRICS_DIR}
  predictions : {PREDS_DIR}    (per-sample, so metrics can be recomputed without re-inference)
  logs        : {LOGS_DIR}
  registry    : {ART}/model_registry.csv
  summary     : {RESULTS_DIR}/run_summary.json
""")

## Done

Read in this order when writing the paper:

1. **`table_gate_report.csv`** — did anything fail? Report failures honestly.
2. **Gate 5 / `rq1_verdict.json`** — RQ1 on both axes (predictive *and* mechanistic).
3. **`table_quantization.csv`** — RQ2: FP32 vs PTQ vs QAT.
4. **`table_05_statistical_tests.csv`** — effect sizes with CIs; a difference is not a claim unless
   the CI excludes zero.
5. **`table_06_external_validation_deepdrid.csv`** — external generalization, reported as measured.

Use `docs/judge.md` Section I's safe phrasing for every claim. Do not describe CSD as successful on
mechanism metrics alone if predictive performance did not move; do not call the model
deployment-ready on the basis of size alone.